# Matching

The purpose of this file is to do one set of matching: **treatment matching**, to match those who graduated from the program against those that did not graduate from the program. 

We will match to examine the following:

-Graduation effect for **earnings** (income)

-Graduation effect for **employment**

-Graduation effect for **criminal justice interaction** (as measured by arrests)

-Graduation effect for **recidivism** (as measured by offenses)

This will be followed (in future analysis) by intent to treat matching of TIP participants with similar non-participants, which we expect will be particularly relevant for criminal justice and recidivism outcomes. 


# Preparation

In [100]:
#Import necessary packages

import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import tqdm
##!pip install linearmodels
from linearmodels import PanelOLS
import statsmodels.api as sm
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.families import Binomial
from statsmodels.genmod.cov_struct import Exchangeable

In [101]:
#Import necessary files

#Student datafile, data up to December 2025 
tip_clean = pd.read_csv(r'TIP_clean.csv', parse_dates=['CreatedDate','DoB', 'Interviewed Only Date', 'InterviewedDate','StartDate','EndDate'])

#Demographic datafile, data up to December 2024
tip_demos = pd.read_csv(r'tip_cohort_demographics_rg.csv')

#Sentencing datafile, data up to December 2024
tip_sentencing = pd.read_csv(r'convictions_matched_to_tip_final.csv', parse_dates = ['DOB', 'GraduatedDate', 'InterviewedDate', 'StartDate','EndDate', 'DOF', 'DOS'])

#Arrests datafile, data up to December 2024
tip_arrests = pd.read_csv(r'arrest_data.csv', parse_dates = ['CaseFiledDate', 'ArrestDate', 'DefendantDOB', 'OffenseDate','OffenseDispositionDate','CaseDispositionDate','InterviewedDate','Orientation Date', 'GraduatedDate', 'CreatedDate','StartDate','EndDate'])

#Earnings datafile, data up to December 2024
tip_earnings = pd.read_csv(r'tip_cohort_uiearnings.csv')

#Deflation information to deflate earnings data
df_inflation = pd.read_csv(r'inflation_data.csv')

#DHS datafile for non-participants
df_dhs = pd.read_csv(r'DHS_demographics.csv')

#full_pcs data
full_pcs = pd.read_csv(r'full_pcs.txt')

#full_earnings data
full_earnings = pd.read_csv(r'ui_earnings.txt')

C:\Users\13429\AppData\Local\Temp\ipykernel_33308\2219236522.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  tip_arrests = pd.read_csv(r'arrest_data.csv', parse_dates = ['CaseFiledDate', 'ArrestDate', 'DefendantDOB', 'OffenseDate','OffenseDispositionDate','CaseDispositionDate','InterviewedDate','Orientation Date', 'GraduatedDate', 'CreatedDate','StartDate','EndDate'])
C:\Users\13429\AppData\Local\Temp\ipykernel_33308\2219236522.py:16: DtypeWarning: Columns (3,6,7,8,9,10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  tip_earnings = pd.read_csv(r'tip_cohort_uiearnings.csv')
C:\Users\13429\AppData\Local\Temp\ipykernel_33308\2219236522.py:25: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  full_pcs = pd.read_csv(r'full_pcs.txt')
C:\Users\13429\AppData\Local\Te

In [102]:
#Clean student datafile column names

all_tip_students = tip_clean.rename(columns={
    'participant_id' : 'tip_id',
    'DoB' : 'DOB',
    'Status' : 'tip_status',
})

all_tip_students.drop(columns=['Unnamed: 25', 'Unnamed: 26'], inplace=True)

#Define statuses as "Graduated" or "Non Graduated" - for the purposes of this matched analysis, we do not care about individuals who do not fall into these statuses

def status(row):
    ts = str(row['tip_status'])
    if ("Graduate" in ts) or ("DNF" in ts and "Orientation" not in ts):
        return 'Participant'
    else:
        return 'Non Participant'

all_tip_students['Status'] = all_tip_students.apply(status, axis=1)
all_tip_students.Status.value_counts()

Status
Participant        1442
Non Participant    1281
Name: count, dtype: int64

In [103]:
# Clean tip_sentencing datafile column names and status
tip_sentencing = tip_sentencing.rename(columns={
    'PCS_OFF_ID': 'pcs_off_id',
    'Status': 'tip_status'
})
tip_sentencing['Status'] = tip_sentencing.apply(status, axis=1)

In [104]:
# -----------------------------
# 1. Create df_participant
# -----------------------------
df_participant = all_tip_students[all_tip_students["Status"] == "Participant"].copy()

# Optional: if you also want the non-participant students from all_tip_students itself
df_non_participant_students = all_tip_students[all_tip_students["Status"] == "Non Participant"].copy()


# -----------------------------
# 2. Check unique non-null pcs_off_id in tip_sentencing
# -----------------------------
tip_pcs_ids = tip_sentencing["pcs_off_id"].dropna().unique()

print("Number of unique non-null pcs_off_id in tip_sentencing:", len(tip_pcs_ids))


# -----------------------------
# 3. Check whether these pcs_off_id have corresponding tip_id
#    (row-level check within tip_sentencing)
# -----------------------------
pcs_tip_check = (
    tip_sentencing.loc[tip_sentencing["pcs_off_id"].notna(), ["pcs_off_id", "tip_id"]]
    .drop_duplicates()
)

# pcs_off_id with at least one non-null tip_id
pcs_with_tip_id = pcs_tip_check.loc[pcs_tip_check["tip_id"].notna(), "pcs_off_id"].unique()

# pcs_off_id that appear but do NOT have any valid tip_id
pcs_without_tip_id = set(tip_pcs_ids) - set(pcs_with_tip_id)

print("Number of unique pcs_off_id with at least one corresponding non-null tip_id:", len(pcs_with_tip_id))
print("Number of unique pcs_off_id without a corresponding non-null tip_id:", len(pcs_without_tip_id))

if len(pcs_without_tip_id) > 0:
    print("pcs_off_id without corresponding tip_id:")
    print(pcs_without_tip_id)
else:
    print("All unique non-null pcs_off_id in tip_sentencing have at least one corresponding non-null tip_id.")


# -----------------------------
# 4. Remove from df_dhs all pcs_off_id that appear in tip_sentencing
#    and create df_non_participant, keep only unique pcs_off_id in df_non_participant
# -----------------------------
df_non_participant = df_dhs[~df_dhs["pcs_off_id"].isin(tip_pcs_ids)].copy()
df_non_participant.drop_duplicates(subset=["pcs_off_id"], inplace=True)

print("Original df_dhs shape:", df_dhs.shape)
print("df_non_participant shape:", df_non_participant.shape)

Number of unique non-null pcs_off_id in tip_sentencing: 1480
Number of unique pcs_off_id with at least one corresponding non-null tip_id: 1480
Number of unique pcs_off_id without a corresponding non-null tip_id: 0
All unique non-null pcs_off_id in tip_sentencing have at least one corresponding non-null tip_id.
Original df_dhs shape: (126232, 6)
df_non_participant shape: (123434, 6)


In [105]:
date_cols = ['DOB', 'StartDate', 'EndDate', 'InterviewedDate', 'Interviewed Only Date']
for col in date_cols:
    if col in df_participant.columns:
        df_participant[col] = pd.to_datetime(df_participant[col], errors = 'coerce')

##Add an indicator for cohort based on StartDate
cutoff = pd.Timestamp("2022-01-01")
df_participant["cohort_2022"] = np.where(
    df_participant["StartDate"].notna(),
    np.where(df_participant["StartDate"] >= cutoff, 1, 0),
    np.nan
)

df_participant["cohort_2022"] = pd.to_numeric(df_participant["cohort_2022"], errors = "coerce")

#Drop anyone with nonsensical ages (under 16)
df_participant = df_participant[df_participant['Age_at_start'] >= 16]

#Add age band based on age at start

def assign_age_band(age):
    if pd.isna(age):
        return np.nan
    elif 18 <= age <= 24:
        return '<=24'
    elif 25 <= age <= 34:
        return '25-34'
    elif age >= 35:
        return '35+'
    else:
        return np.nan

df_participant['age_band'] = df_participant['Age_at_start'].apply(assign_age_band)

df_participant.head()

,tip_id,CreatedDate,DOB,tip_status,Course,StartDate,EndDate,InterviewedDate,Interviewed Only Date,DriversLicense,...,EndInternet,AccessResume,EndResume,License To Thrive,Workforce Housing,SNAP Assistance,Age_at_start,Status,cohort_2022,age_band
0,1,2018-07-11 00:00:00+00:00,1992-08-26,DNF: Dropped Out,Introduction to Masonry,2018-07-12,2018-08-01 00:00:00+00:00,NaT,NaT,No,...,NaN,NaN,NaN,NaN,NaN,NaN,25.840055,Participant,0.0,25-34
1,2,2018-07-11 00:00:00+00:00,1997-01-27,DNF: Asked to Leave,Introduction to Masonry,2018-07-12,NaT,NaT,NaT,No,...,NaN,NaN,NaN,NaN,NaN,NaN,21.424470,Participant,0.0,<=24
2,3,2018-07-11 00:00:00+00:00,1999-05-04,DNF: Asked to Leave,Introduction to Masonry,2018-07-23,NaT,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,19.193438,Participant,0.0,<=24
3,4,2018-07-12 00:00:00+00:00,1982-08-15,Graduated-Employed,Introduction to Masonry,2013-08-12,2013-10-16 00:00:00+00:00,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,30.950103,Participant,0.0,25-34
4,5,2018-07-12 00:00:00+00:00,1988-04-10,Graduated-Employed,Introduction to Masonry,2010-11-29,2011-02-04 00:00:00+00:00,NaT,NaT,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,22.605605,Participant,0.0,<=24


In [106]:
#Merge with demographics

tip_demos['tip_id'] = pd.to_numeric(tip_demos['tip_id'], errors='coerce')
df_participant['tip_id'] = pd.to_numeric(df_participant['tip_id'], errors='coerce')

df_participant = df_participant.merge(
    tip_demos,
    on = 'tip_id',
    how = 'left'
)

In [107]:
# Prepare sentencing data
tip_sentencing['DOB'] = pd.to_datetime(tip_sentencing['DOB'], errors='coerce').dt.tz_localize(None)
tip_sentencing['DOF'] = pd.to_datetime(tip_sentencing['DOF'], errors='coerce').dt.tz_localize(None)
tip_sentencing['DOS'] = pd.to_datetime(tip_sentencing['DOS'], errors='coerce').dt.tz_localize(None)
tip_sentencing['StartDate'] = pd.to_datetime(tip_sentencing['StartDate'], errors='coerce').dt.tz_localize(None)
tip_sentencing['EndDate'] = pd.to_datetime(tip_sentencing['EndDate'], errors='coerce').dt.tz_localize(None)
tip_sentencing['tip_id'] = pd.to_numeric(tip_sentencing['tip_id'], errors='coerce')
tip_sentencing['OGS'] = pd.to_numeric(tip_sentencing['OGS'], errors='coerce')

tip_sentencing['age_at_sentencing'] = (
    (tip_sentencing['DOS'] - tip_sentencing['DOB']).dt.days // 365
)

tip_sentencing['age_at_offense'] = (
    (tip_sentencing['DOF'] - tip_sentencing['DOB']).dt.days // 365
)

# merge student StartDate for consistent baseline cutoff
tip_sentencing = tip_sentencing.merge(
    df_participant[['tip_id', 'StartDate']],
    on='tip_id',
    how='left',
    suffixes=('', '_student')
)

if 'StartDate_student' in tip_sentencing.columns:
    tip_sentencing['StartDate'] = tip_sentencing['StartDate_student'].combine_first(tip_sentencing['StartDate'])
    tip_sentencing = tip_sentencing.drop(columns=['StartDate_student'])

# before TIP start: sentencing-based summary
sent_before = tip_sentencing[
    tip_sentencing['DOS'].notna() &
    tip_sentencing['StartDate'].notna() &
    (tip_sentencing['DOS'] < tip_sentencing['StartDate'])
].copy()

sent_summary = sent_before.groupby('tip_id').agg(
    num_sentences_before_start=('DOS', 'count'),
    any_sentence_before_start=('DOS', lambda x: 1 if len(x) > 0 else 0),
    min_age_at_sentencing_before_start=('age_at_sentencing', 'min'),
    max_ogs_before_start=('OGS', 'max'),
    any_high_ogs_before_start=('OGS', lambda x: 1 if (x > 5).any() else 0)
).reset_index()

# create wide table
sent_wide = tip_sentencing[['tip_id', 'age_at_sentencing']].drop_duplicates()
sent_wide['value'] = 1
sent_wide = sent_wide.pivot(index='tip_id', columns='age_at_sentencing', values='value').fillna(0)
sent_wide = sent_wide.reset_index().rename_axis(None, axis=1)

drop_cols = [np.nan]
for column in sent_wide.columns[2:]:
    column = int(column)
    new_col = f'{column}_sentencing_age'
    sent_wide[new_col] = sent_wide[column]
    drop_cols.append(column)

sent_wide = sent_wide.drop(columns=drop_cols)

# create a high OGS table
high_tip_ogs = tip_sentencing[tip_sentencing['OGS'] > 5]
high_tip_ogs_wide = high_tip_ogs[['tip_id', 'age_at_sentencing']].drop_duplicates().dropna()
high_tip_ogs_wide['value'] = 1
high_tip_ogs_wide = high_tip_ogs_wide.pivot(index='tip_id', columns='age_at_sentencing', values='value').fillna(0)
high_tip_ogs_wide = high_tip_ogs_wide.reset_index().rename_axis(None, axis=1)

drop_cols = []
for column in high_tip_ogs_wide.columns[1:]:
    column = int(column)
    new_col = f'{column}_sentencing_age_high_ogs'
    high_tip_ogs_wide[new_col] = high_tip_ogs_wide[column]
    drop_cols.append(column)

high_tip_ogs_wide = high_tip_ogs_wide.drop(columns=drop_cols)

# merge both wide tables together
sent_wide = sent_wide.merge(
    high_tip_ogs_wide,
    on='tip_id',
    how='left'
).fillna(0)

sent_wide.head()

,tip_id,16_sentencing_age,17_sentencing_age,18_sentencing_age,19_sentencing_age,20_sentencing_age,21_sentencing_age,22_sentencing_age,23_sentencing_age,24_sentencing_age,...,43_sentencing_age_high_ogs,44_sentencing_age_high_ogs,45_sentencing_age_high_ogs,46_sentencing_age_high_ogs,47_sentencing_age_high_ogs,48_sentencing_age_high_ogs,49_sentencing_age_high_ogs,51_sentencing_age_high_ogs,52_sentencing_age_high_ogs,58_sentencing_age_high_ogs
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [108]:
# Build combined offense-date table using sentencing DOF + arrests OffenseDate
tip_arrests['tip_id'] = pd.to_numeric(tip_arrests['tip_id'], errors='coerce')

# parse mixed-format OffenseDate
tip_arrests['OffenseDate'] = (
    tip_arrests['OffenseDate']
    .astype(str)
    .str.strip()
    .replace({'': np.nan, 'nan': np.nan, 'NaT': np.nan})
)

tip_arrests['OffenseDate'] = pd.to_datetime(
    tip_arrests['OffenseDate'],
    errors='coerce',
    format='mixed'
).dt.tz_localize(None)

# merge participant DOB and StartDate onto arrests
tip_arrests = tip_arrests.merge(
    df_participant[['tip_id', 'DOB', 'StartDate']],
    on='tip_id',
    how='left',
    suffixes=('', '_student')
)

if 'StartDate_student' in tip_arrests.columns:
    tip_arrests['StartDate'] = tip_arrests['StartDate_student'].combine_first(tip_arrests['StartDate'])
    tip_arrests = tip_arrests.drop(columns=['StartDate_student'])

tip_arrests['DOB'] = pd.to_datetime(tip_arrests['DOB'], errors='coerce').dt.tz_localize(None)
tip_arrests['StartDate'] = pd.to_datetime(tip_arrests['StartDate'], errors='coerce').dt.tz_localize(None)

# offense dates already present in sentencing
sent_offense_dates = (
    tip_sentencing[['tip_id', 'DOF', 'DOB', 'StartDate']]
    .dropna(subset=['tip_id', 'DOF'])
    .drop_duplicates()
    .rename(columns={'DOF': 'offense_date'})
)

sent_offense_dates['source'] = 'sentencing'

# offense dates from arrests
arrest_offense_dates = (
    tip_arrests[['tip_id', 'OffenseDate', 'DOB', 'StartDate']]
    .dropna(subset=['tip_id', 'OffenseDate'])
    .drop_duplicates()
    .rename(columns={'OffenseDate': 'offense_date'})
)

arrest_offense_dates['source'] = 'arrest'

# append and deduplicate, keeping sentencing source first if both exist
combined_offense_dates = pd.concat(
    [sent_offense_dates, arrest_offense_dates],
    ignore_index=True
)

combined_offense_dates['source_rank'] = combined_offense_dates['source'].map({
    'sentencing': 0,
    'arrest': 1
})

combined_offense_dates = (
    combined_offense_dates
    .sort_values(['tip_id', 'offense_date', 'source_rank'])
    .drop_duplicates(subset=['tip_id', 'offense_date'], keep='first')
    .drop(columns=['source_rank'])
    .reset_index(drop=True)
)

combined_offense_dates['age_at_offense'] = (
    (combined_offense_dates['offense_date'] - combined_offense_dates['DOB']).dt.days // 365
)

combined_offense_dates.head()

,tip_id,offense_date,DOB,StartDate,source,age_at_offense
0,1.0,2001-09-16,1992-08-26,2018-07-12,arrest,9.0
1,1.0,2016-01-09,1992-08-26,2018-07-12,sentencing,23.0
2,2.0,2010-11-16,1997-01-27,2018-07-12,arrest,13.0
3,2.0,2017-05-02,1997-01-27,2018-07-12,sentencing,20.0
4,3.0,2018-09-14,1999-05-04,2018-07-23,arrest,19.0


In [109]:
# Offense summary before TIP start using combined offense dates
off_before = combined_offense_dates[
    combined_offense_dates['offense_date'].notna() &
    combined_offense_dates['StartDate'].notna() &
    (combined_offense_dates['offense_date'] < combined_offense_dates['StartDate'])
].copy()

off_summary = off_before.groupby('tip_id').agg(
    num_offenses_before_start=('offense_date', 'count'),
    any_offense_before_start=('offense_date', lambda x: 1 if x.count() > 0 else 0),
    age_of_first_offense=('age_at_offense', 'min')
).reset_index()

off_summary.head()

,tip_id,num_offenses_before_start,any_offense_before_start,age_of_first_offense
0,1.0,2,1,9.0
1,2.0,2,1,13.0
2,7.0,1,1,19.0
3,8.0,2,1,20.0
4,10.0,1,1,18.0


In [110]:
# Prepare arrests data
tip_arrests['ArrestDate'] = pd.to_datetime(tip_arrests['ArrestDate'], errors='coerce').dt.tz_localize(None)
tip_arrests['tip_id'] = pd.to_numeric(tip_arrests['tip_id'], errors='coerce')

tip_arrests = tip_arrests.merge(
    df_participant[['tip_id', 'DOB', 'StartDate']],
    on='tip_id',
    how='left',
    suffixes=('', '_student')
)

if 'StartDate_student' in tip_arrests.columns:
    tip_arrests['StartDate'] = tip_arrests['StartDate_student'].combine_first(tip_arrests['StartDate'])
    tip_arrests = tip_arrests.drop(columns=['StartDate_student'])

tip_arrests['age_at_arrest'] = (
    (tip_arrests['ArrestDate'] - tip_arrests['DOB']).dt.days // 365
)

arrests_before = tip_arrests[
    tip_arrests['ArrestDate'].notna() &
    tip_arrests['StartDate'].notna() &
    (tip_arrests['ArrestDate'] < tip_arrests['StartDate'])
].copy()

arrest_summary = arrests_before.groupby('tip_id').agg(
    num_arrests_before_start=('ArrestDate', 'count'),
    any_arrest_before_start=('ArrestDate', lambda x: 1 if len(x) > 0 else 0),
    min_age_at_arrest_before_start=('age_at_arrest', 'min')
).reset_index()

# pivot wider
arrests_wide = tip_arrests[['tip_id', 'age_at_arrest']].drop_duplicates()
arrests_wide['value'] = 1
arrests_wide = arrests_wide.pivot(index='tip_id', columns='age_at_arrest', values='value').fillna(0).reset_index().rename_axis(None, axis=1)

drop_cols = [np.nan]
for column in arrests_wide.columns[2:]:
    column = int(column)
    new_col = f'{column}_arrest_age'
    arrests_wide[new_col] = arrests_wide[column]
    drop_cols.append(column)

arrests_wide = arrests_wide.drop(columns=drop_cols)

# add YearsSinceTIP to long df for later outcome analysis
tip_arrests['YearsSinceTIP'] = (tip_arrests['ArrestDate'] - tip_arrests['StartDate']).dt.days / 365

In [111]:
#Prepare earnings data

# process quarter to datetime
q_str = tip_earnings["quarter"].astype(str).str.strip()

tip_earnings["year"] = q_str.str[:4].astype(int)
tip_earnings["qnum"] = q_str.str[-1].astype(int)  # 1-4

# map to calendar quarter end month/day
# Q1 -> 03/31
# Q2 -> 06/30
# Q3 -> 09/30
# Q4 -> 12/31 

#Nikki: This was slightly different in your code. I matched it to what we did for the midterm analysis.
month_map = {1: 3, 2: 6, 3: 9, 4: 12}
day_map   = {1: 31, 2: 30, 3: 30, 4: 31}

tip_earnings["cal_end_year"] = tip_earnings["year"]
tip_earnings["cal_end_month"] = tip_earnings["qnum"].map(month_map)
tip_earnings["cal_end_day"] = tip_earnings["qnum"].map(day_map)

tip_earnings["year_quarter_dt"] = pd.to_datetime(
    dict(
        year=tip_earnings["cal_end_year"],
        month=tip_earnings["cal_end_month"],
        day=tip_earnings["cal_end_day"],
    ),
    errors="coerce",
)

tip_earnings["year_month"] = tip_earnings["year_quarter_dt"].dt.to_period("M")
tip_earnings.drop(columns=["Unnamed: 0", "qnum", "cal_end_year", "cal_end_month", "cal_end_day"], errors="ignore", inplace=True)

In [112]:
#Adjust earnings data for inflation (deflate the data)
df_inflation.drop(columns=['HALF1', 'HALF2'], inplace=True)
df_inflation = df_inflation.melt(id_vars='Year', var_name='Month', value_name='CPI')

month_map = {'Jan':'01','Feb':'02','Mar':'03','Apr':'04','May':'05','Jun':'06',
             'Jul':'07','Aug':'08','Sep':'09','Oct':'10','Nov':'11','Dec':'12'}
df_inflation["year_month"] = (
    df_inflation["Year"].astype(str) + "-" + df_inflation["Month"].map(month_map)
)
# sort by date to ensure that the latest CPI value is at the end of the DataFrame
df_inflation = df_inflation.sort_values("year_month").reset_index(drop=True)
latest_cpi = df_inflation.loc[df_inflation["year_month"] == "2025-12", "CPI"].iloc[0]

tip_earnings["year_month"] = tip_earnings["year_month"].astype(str)

cpi_map = df_inflation.drop_duplicates("year_month").set_index("year_month")["CPI"]
tip_earnings["CPI"] = tip_earnings["year_month"].map(cpi_map)

tip_earnings["adjusted_earnings"] = tip_earnings["earnings"] * (latest_cpi / tip_earnings["CPI"])

In [113]:
##create adjusted earnings that is ready for log
tip_earnings["adjusted_earnings_plus_one"] = tip_earnings["adjusted_earnings"] + 1

#Calculate earnings 1 year and 2 years pre-TIP for match
tip_earnings = pd.merge(
    tip_earnings,
    df_participant[['tip_id', 'StartDate']].drop_duplicates(),
    on='tip_id',
    how = 'left'
)
    
# keep non-missing earnings, but DO NOT drop zeros
tip_earnings = tip_earnings.dropna(subset=['earnings', 'StartDate']).copy()

# time distance relative to TIP start
tip_earnings['quarters_from_start'] = (
    (tip_earnings['year_quarter_dt'].dt.year - tip_earnings['StartDate'].dt.year) * 4
    + (tip_earnings['year_quarter_dt'].dt.quarter - tip_earnings['StartDate'].dt.quarter)
)

tip_earnings.head()

#aggregate 1-year pre-TIP earnings (quarters -1 through -4)
earnings_1yr_pre = (
    tip_earnings[tip_earnings['quarters_from_start'].between(-4, -1)]
    .groupby('tip_id')['adjusted_earnings']
    .sum()
    .reset_index()
    .rename(columns={'adjusted_earnings': 'earnings_1yr_preTIP'})
)
##Consider cumulative - double check this isn't what the previous team was doing
#2-year pre-TIP earnings (quarters -5 through -8)
earnings_2yr_pre = (
    tip_earnings[tip_earnings['quarters_from_start'].between(-8, -5)]
    .groupby('tip_id')['adjusted_earnings']
    .sum()
    .reset_index()
    .rename(columns={'adjusted_earnings': 'earnings_2yr_preTIP'})
)

earnings_2yr_pre.head()

,tip_id,earnings_2yr_preTIP
0,1,410.102835
1,2,8049.772420
2,29,35597.891193
3,210,3436.631229
4,216,1467.860077


In [114]:
#Summarize employment immediately pre-TIP using earnings data (just checks for existence of any earnings in the year immediately prior to TIP
job_right_before = (
    tip_earnings[tip_earnings['quarters_from_start'].between(-4, -1)]  #Nikki I adjusted this to quarters 1-4 instead of just quarter 1
    .groupby('tip_id')['adjusted_earnings']
    .apply(lambda x: 1 if (x > 0).any() else 0)
    .reset_index(name='job_right_before_tip')
)

In [115]:
#Calculate age at point of each earnings record and pivot wide

tip_dob = df_participant[['tip_id', 'DOB']].drop_duplicates()
tip_earnings = pd.merge(
    tip_earnings,
    tip_dob,
    on='tip_id',
    how='left'
)

tip_earnings['age'] = (tip_earnings['year_quarter_dt']-tip_earnings['DOB']).dt.days//365

#pivot wider
earnings_wide = tip_earnings[['tip_id','age', 'adjusted_earnings']].copy()
earnings_wide = earnings_wide.groupby(['tip_id', 'age'])['adjusted_earnings'].sum().reset_index()
earnings_wide = earnings_wide.pivot(index='tip_id', columns='age', values='adjusted_earnings').fillna(0)
earnings_wide = earnings_wide.reset_index().rename_axis(None, axis=1)
earnings_wide.rename(columns={col: f'{col}_earnings' for col in earnings_wide.columns[1:]}, inplace=True)

##Note: Some records are for individuals that were apparently 10-13 at the time of the earnings record. Maybe a mismatch. For safety we've removed nonsensical ages from the dataframe based on age at start.
earnings_wide.head()

#add YearsSinceTIP to long df for later outcome analysis
tip_earnings['YearsSinceTIP'] = (tip_earnings['year_quarter_dt'] - tip_earnings['StartDate']).dt.days / 365

In [116]:
tip_earnings[tip_earnings['age'] < 16][['tip_id', 'DOB', 'year_quarter_dt', 'adjusted_earnings', 'age']].head()

,tip_id,DOB,year_quarter_dt,adjusted_earnings,age
32,1292,2002-02-19,2017-12-31,1263.227491,15
33,1292,2002-02-19,2017-12-31,946.434749,15
40,1641,2002-10-11,2017-12-31,10437.072090,15
41,1641,2002-10-11,2017-12-31,1635.228927,15
42,2246,2002-03-17,2017-12-31,8294.448979,15


In [117]:
#Create one wide dataframe

all_tip_data = pd.merge(
    df_participant,
    sent_wide.fillna(0),
    on = 'tip_id',
    how = 'left'
)

all_tip_data = pd.merge(
    all_tip_data,
    arrests_wide.fillna(0),
    on = 'tip_id',
    how = 'left'
)

all_tip_data = pd.merge(
    all_tip_data,
    earnings_wide.fillna(0),
    on = 'tip_id',
    how = 'left'
)

In [118]:
#fill remaining NAs (for unmatched IDs) with 0s
all_tip_data[sent_wide.columns] = all_tip_data[sent_wide.columns].fillna(0)
all_tip_data[arrests_wide.columns] = all_tip_data[arrests_wide.columns].fillna(0)
all_tip_data[earnings_wide.columns] = all_tip_data[earnings_wide.columns].fillna(0)
all_tip_data[tip_demos.columns] = all_tip_data[tip_demos.columns].fillna(0)

all_tip_data.head()

,tip_id,CreatedDate,DOB,tip_status,Course,StartDate,EndDate,InterviewedDate,Interviewed Only Date,DriversLicense,...,56_earnings,57_earnings,58_earnings,59_earnings,60_earnings,61_earnings,62_earnings,63_earnings,64_earnings,65_earnings
0,1,2018-07-11 00:00:00+00:00,1992-08-26,DNF: Dropped Out,Introduction to Masonry,2018-07-12,2018-08-01 00:00:00+00:00,NaT,NaT,No,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,2018-07-11 00:00:00+00:00,1997-01-27,DNF: Asked to Leave,Introduction to Masonry,2018-07-12,NaT,NaT,NaT,No,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,2018-07-11 00:00:00+00:00,1999-05-04,DNF: Asked to Leave,Introduction to Masonry,2018-07-23,NaT,NaT,NaT,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,2018-07-12 00:00:00+00:00,1982-08-15,Graduated-Employed,Introduction to Masonry,2013-08-12,2013-10-16 00:00:00+00:00,NaT,NaT,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,2018-07-12 00:00:00+00:00,1988-04-10,Graduated-Employed,Introduction to Masonry,2010-11-29,2011-02-04 00:00:00+00:00,NaT,NaT,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [119]:
#combine with summary data for matching
all_tip_data = pd.merge(
    all_tip_data,
    sent_summary, 
    on='tip_id', 
    how='left'
)

all_tip_data = pd.merge(
    all_tip_data,
    arrest_summary, 
    on='tip_id', 
    how='left'
)

all_tip_data = pd.merge(
    all_tip_data,
    earnings_1yr_pre, 
    on='tip_id', 
    how='left'
)

all_tip_data = pd.merge(
    all_tip_data,
    earnings_2yr_pre, 
    on='tip_id', 
    how='left'
)

all_tip_data = pd.merge(
    all_tip_data,
    job_right_before, 
    on='tip_id', 
    how='left'
)

all_tip_data = pd.merge(
    all_tip_data,
    off_summary,
    on='tip_id',
    how='left'
)

tip_fill_zero_cols = [
    'num_sentences_before_start',
    'any_sentence_before_start',
    'max_ogs_before_start',
    'any_high_ogs_before_start',
    'num_offenses_before_start',
    'any_offense_before_start',
    'num_arrests_before_start',
    'any_arrest_before_start',
    'earnings_1yr_preTIP',
    'earnings_2yr_preTIP',
    'job_right_before_tip'
]

for col in tip_fill_zero_cols:
    if col in all_tip_data.columns:
        all_tip_data[col] = all_tip_data[col].fillna(0)

In [120]:
tip_arrests['tip_id'].nunique()

1098

In [121]:
all_tip_data["age_of_first_offense"].notna().value_counts()

age_of_first_offense
False    747
True     668
Name: count, dtype: int64

In [122]:
print("all_tip_data shape:", all_tip_data.shape)
print("\nTIP missingness summary:")
print(
    all_tip_data[
        [
            'tip_id', 'DOB', 'StartDate', 'EndDate','Status', 'cohort_2022',
            'Age_at_start', 'age_band', 'gender', 'race', 'census_tract',
            'num_sentences_before_start', 'any_sentence_before_start',
            'max_ogs_before_start', 'any_high_ogs_before_start', 
            'num_offenses_before_start', 'any_offense_before_start', 'age_of_first_offense', 
            'num_arrests_before_start', 'any_arrest_before_start',
            'earnings_1yr_preTIP', 'earnings_2yr_preTIP',
            'job_right_before_tip'
        ]
    ].isna().sum()
)

all_tip_data shape: (1415, 230)

TIP missingness summary:
tip_id                          0
DOB                             0
StartDate                       0
EndDate                        17
Status                          0
cohort_2022                     0
Age_at_start                    0
age_band                      116
gender                          0
race                            0
census_tract                    0
num_sentences_before_start      0
any_sentence_before_start       0
max_ogs_before_start            0
any_high_ogs_before_start       0
num_offenses_before_start       0
any_offense_before_start        0
age_of_first_offense          747
num_arrests_before_start        0
any_arrest_before_start         0
earnings_1yr_preTIP             0
earnings_2yr_preTIP             0
job_right_before_tip            0
dtype: int64


## Non Participant

In [123]:
# --------------------------------------------------
# Block 1: Standardize source tables
# --------------------------------------------------

# Standardize ids
df_non_participant["pcs_off_id"] = pd.to_numeric(df_non_participant["pcs_off_id"], errors="coerce")
full_pcs["PCS_OFF_ID"] = pd.to_numeric(full_pcs["PCS_OFF_ID"], errors="coerce")
full_earnings["pcs_off_id"] = pd.to_numeric(full_earnings["pcs_off_id"], errors="coerce")

# Standardize dates in PCS
full_pcs["DOB"] = pd.to_datetime(full_pcs["DOB"], errors="coerce")
full_pcs["DOF"] = pd.to_datetime(full_pcs["DOF"], errors="coerce")
full_pcs["DOS"] = pd.to_datetime(full_pcs["DOS"], errors="coerce")

# Standardize year variables in df_non_participant
if "year_of_birth" in df_non_participant.columns:
    df_non_participant["year_of_birth"] = pd.to_numeric(df_non_participant["year_of_birth"], errors="coerce")

if "year_of_death" in df_non_participant.columns:
    df_non_participant["year_of_death"] = pd.to_numeric(df_non_participant["year_of_death"], errors="coerce")

# Restrict PCS / earnings to ids in non-participant pool
non_participant_ids = df_non_participant["pcs_off_id"].dropna().unique()

pcs_np = full_pcs[full_pcs["PCS_OFF_ID"].isin(non_participant_ids)].copy()
earnings_np = full_earnings[full_earnings["pcs_off_id"].isin(non_participant_ids)].copy()

print("Non-participant unique ids in df_non_participant:", df_non_participant["pcs_off_id"].nunique())
print("Matched unique ids in full_pcs:", pcs_np["PCS_OFF_ID"].nunique())
print("Matched unique ids in full_earnings:", earnings_np["pcs_off_id"].nunique())

Non-participant unique ids in df_non_participant: 123433
Matched unique ids in full_pcs: 123433
Matched unique ids in full_earnings: 103625


In [124]:
# --------------------------------------------------
# Block 2: Build static person-level base table for exact matching
# --------------------------------------------------

def first_nonnull(series):
    s = series.dropna()
    return s.iloc[0] if len(s) > 0 else np.nan

# First offense information from PCS
first_offense_info = (
    pcs_np[pcs_np["DOF"].notna()]
    .sort_values(["PCS_OFF_ID", "DOF"])
    .groupby("PCS_OFF_ID", as_index=False)
    .first()[["PCS_OFF_ID", "DOF", "DOB"]]
    .rename(columns={
        "PCS_OFF_ID": "pcs_off_id",
        "DOF": "first_offense_date",
        "DOB": "dob_from_first_offense_record"
    })
)

# Static summary from PCS
pcs_person_static = (
    pcs_np.groupby("PCS_OFF_ID", as_index=False)
    .agg(
        dob_from_pcs=("DOB", "min"),
        off_race_from_pcs=("OFF_RACE", first_nonnull),
        off_sex_from_pcs=("OFF_SEX", first_nonnull),
        total_offense_records=("DOF", lambda x: x.notna().sum()),   # row count, not unique DOF
        total_sentencing_records=("DOS", lambda x: x.notna().sum()),
        max_ogs_observed=("OGS", "max"),
        any_high_ogs_observed=("OGS", lambda x: 1 if (pd.to_numeric(x, errors="coerce") > 5).any() else 0),
        first_observed_dof=("DOF", "min"),
        last_observed_dof=("DOF", "max"),
        first_observed_dos=("DOS", "min"),
        last_observed_dos=("DOS", "max"),
    )
    .rename(columns={"PCS_OFF_ID": "pcs_off_id"})
)

# Merge into non-participant base
np_static_base = df_non_participant.merge(
    pcs_person_static,
    on="pcs_off_id",
    how="left"
).merge(
    first_offense_info,
    on="pcs_off_id",
    how="left"
)

# Final DOB: prefer PCS DOB
np_static_base["DOB"] = np_static_base["dob_from_pcs"]

# Fallback DOB from year_of_birth if needed
np_static_base["DOB_fallback"] = pd.to_datetime(
    np.where(
        np_static_base["DOB"].isna() & np_static_base["year_of_birth"].notna(),
        np_static_base["year_of_birth"].astype("Int64").astype(str) + "-01-01",
        pd.NaT
    ),
    errors="coerce"
)

np_static_base["DOB_final"] = np_static_base["DOB"].combine_first(np_static_base["DOB_fallback"])

# Final year_of_birth
np_static_base["year_of_birth_final"] = np.where(
    np_static_base["DOB_final"].notna(),
    np_static_base["DOB_final"].dt.year,
    np_static_base["year_of_birth"]
)

# Calendar age of first offense
np_static_base["age_of_first_offense"] = (
    (np_static_base["first_offense_date"] - np_static_base["DOB_final"]).dt.days // 365
)

# Keep clean columns
np_static_base = np_static_base.drop(
    columns=["dob_from_pcs", "DOB", "DOB_fallback", "dob_from_first_offense_record"],
    errors="ignore"
).rename(columns={"DOB_final": "DOB"})

print("np_static_base shape:", np_static_base.shape)
np_static_base[[
    "pcs_off_id", "DOB", "year_of_birth_final", "first_offense_date",
    "age_of_first_offense", "gender", "race"
]].head()

np_static_base shape: (123434, 20)


,pcs_off_id,DOB,year_of_birth_final,first_offense_date,age_of_first_offense,gender,race
0,879948.0,1989-10-02,1989.0,2018-07-27,28.0,Female,Other
1,885404.0,1995-12-02,1995.0,2019-02-01,23.0,Female,White
2,1191002.0,1951-05-13,1951.0,NaT,NaN,Female,White
3,1315519.0,1975-02-07,1975.0,2000-11-25,25.0,Male,White
4,495537.0,1989-03-07,1989.0,2018-02-18,28.0,Male,NaN


In [125]:
# --------------------------------------------------
# Block 3: Build PCS long table for later dynamic calculations
# --------------------------------------------------

pcs_long_np = pcs_np.copy().rename(columns={"PCS_OFF_ID": "pcs_off_id"})

keep_cols = [
    "pcs_off_id", "DOB", "DOF", "DOS", "OGS", "INCMIN", "OFF_RACE", "OFF_SEX"
]
pcs_long_np = pcs_long_np[[col for col in keep_cols if col in pcs_long_np.columns]].copy()

# numeric OGS
if "OGS" in pcs_long_np.columns:
    pcs_long_np["OGS"] = pd.to_numeric(pcs_long_np["OGS"], errors="coerce")

# Merge cleaned DOB
pcs_long_np = pcs_long_np.merge(
    np_static_base[["pcs_off_id", "DOB"]].drop_duplicates(),
    on="pcs_off_id",
    how="left",
    suffixes=("", "_base")
)

if "DOB_base" in pcs_long_np.columns:
    pcs_long_np["DOB"] = pcs_long_np["DOB_base"].combine_first(pcs_long_np["DOB"])
    pcs_long_np = pcs_long_np.drop(columns=["DOB_base"], errors="ignore")

# Add age variables
pcs_long_np["age_at_offense"] = (
    (pcs_long_np["DOF"] - pcs_long_np["DOB"]).dt.days // 365
)
pcs_long_np["age_at_sentencing"] = (
    (pcs_long_np["DOS"] - pcs_long_np["DOB"]).dt.days // 365
)

# Helper flags
pcs_long_np["has_offense_record"] = np.where(pcs_long_np["DOF"].notna(), 1, 0)
pcs_long_np["has_sentencing_record"] = np.where(pcs_long_np["DOS"].notna(), 1, 0)
pcs_long_np["high_ogs_flag"] = np.where(pcs_long_np["OGS"] > 5, 1, 0)

# Sort for later anchor-based calculations
pcs_long_np = pcs_long_np.sort_values(["pcs_off_id", "DOF", "DOS"]).reset_index(drop=True)

print("pcs_long_np shape:", pcs_long_np.shape)
pcs_long_np.head()

pcs_long_np shape: (348019, 13)


,pcs_off_id,DOB,DOF,DOS,OGS,INCMIN,OFF_RACE,OFF_SEX,age_at_offense,age_at_sentencing,has_offense_record,has_sentencing_record,high_ogs_flag
0,115.0,1966-02-03,2019-01-29,2022-10-26,5.0,NaN,White,M,53.0,56,1,1,0
1,209.0,1962-05-03,2019-04-16,2020-01-02,1.0,NaN,White,M,56.0,57,1,1,0
2,209.0,1962-05-03,2019-04-16,2020-01-02,1.0,0.197368,White,M,56.0,57,1,1,0
3,328.0,1982-02-02,2022-02-15,2022-09-21,1.0,NaN,White,M,40.0,40,1,1,0
4,423.0,1963-01-12,2021-12-27,2022-10-24,5.0,NaN,White,M,58.0,59,1,1,0


In [126]:
# --------------------------------------------------
# Block 4: Build earnings long table
# --------------------------------------------------

q_str = earnings_np["quarter"].astype(str).str.strip()

earnings_np["year"] = q_str.str[:4].astype(int)
earnings_np["qnum"] = q_str.str[-1].astype(int)

month_map = {1: 3, 2: 6, 3: 9, 4: 12}
day_map = {1: 31, 2: 30, 3: 30, 4: 31}

earnings_np["year_quarter_dt"] = pd.to_datetime(
    dict(
        year=earnings_np["year"],
        month=earnings_np["qnum"].map(month_map),
        day=earnings_np["qnum"].map(day_map),
    ),
    errors="coerce"
)

earnings_np["year_month"] = earnings_np["year_quarter_dt"].dt.to_period("M").astype(str)

earnings_long_np = earnings_np.copy()
earnings_long_np["CPI"] = earnings_long_np["year_month"].map(cpi_map)
earnings_long_np["adjusted_earnings"] = earnings_long_np["earnings"] * (latest_cpi / earnings_long_np["CPI"])
earnings_long_np["adjusted_earnings_plus_one"] = earnings_long_np["adjusted_earnings"] + 1

# Merge cleaned DOB
earnings_long_np = earnings_long_np.merge(
    np_static_base[["pcs_off_id", "DOB"]].drop_duplicates(),
    on="pcs_off_id",
    how="left"
)

earnings_long_np["age_at_earnings"] = (
    (earnings_long_np["year_quarter_dt"] - earnings_long_np["DOB"]).dt.days // 365
)

earnings_long_np = earnings_long_np.sort_values(["pcs_off_id", "year_quarter_dt"]).reset_index(drop=True)

print("earnings_long_np shape:", earnings_long_np.shape)
earnings_long_np.head()

earnings_long_np shape: (2973176, 20)


,pcs_off_id,quarter,employer_legal_name,earnings,naics_code,naics_sector_2017,naics_subsector_2017,naics_industry_2017,naics_sector_2022,naics_subsector_2022,naics_industry_2022,year,qnum,year_quarter_dt,year_month,CPI,adjusted_earnings,adjusted_earnings_plus_one,DOB,age_at_earnings
0,115.0,20173,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017,3,2017-09-30,2017-09,246.819,0.0,1.0,1966-02-03,51
1,115.0,20174,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017,4,2017-12-31,2017-12,246.524,0.0,1.0,1966-02-03,51
2,115.0,20181,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018,1,2018-03-31,2018-03,249.554,0.0,1.0,1966-02-03,52
3,115.0,20182,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018,2,2018-06-30,2018-06,251.989,0.0,1.0,1966-02-03,52
4,115.0,20183,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018,3,2018-09-30,2018-09,252.439,0.0,1.0,1966-02-03,52


In [127]:
# --------------------------------------------------
# Block 5: Build static earnings coverage summary
# --------------------------------------------------

earnings_coverage_np = (
    earnings_long_np.groupby("pcs_off_id", as_index=False)
    .agg(
        first_earnings_dt=("year_quarter_dt", "min"),
        last_earnings_dt=("year_quarter_dt", "max"),
        total_earnings_records=("year_quarter_dt", "count"),
        total_observed_earnings=("adjusted_earnings", "sum"),
        ever_positive_earnings=("adjusted_earnings", lambda x: 1 if (x > 0).any() else 0)
    )
)

np_static_base = np_static_base.merge(
    earnings_coverage_np,
    on="pcs_off_id",
    how="left"
)

print("np_static_base shape after earnings coverage merge:", np_static_base.shape)

np_static_base shape after earnings coverage merge: (123434, 25)


In [128]:
# --------------------------------------------------
# Block 6: Final objects for matching
# --------------------------------------------------

# Static exact-match base
nonparticipant_static_base = np_static_base.copy()

# Long PCS records for dynamic pre-start and post-start calculations
nonparticipant_pcs_long = pcs_long_np.copy()

# Long earnings records for dynamic pre-start and post-start calculations
nonparticipant_earnings_long = earnings_long_np.copy()

print("nonparticipant_static_base:", nonparticipant_static_base.shape)
print("nonparticipant_pcs_long:", nonparticipant_pcs_long.shape)
print("nonparticipant_earnings_long:", nonparticipant_earnings_long.shape)

nonparticipant_static_base: (123434, 25)
nonparticipant_pcs_long: (348019, 13)
nonparticipant_earnings_long: (2973176, 20)


In [129]:
# --------------------------------------------------
# Block 7: Quick checks for exact-match variables
# --------------------------------------------------

print("\nMissing DOB:")
print(nonparticipant_static_base["DOB"].isna().sum())

print("\nMissing year_of_birth_final:")
print(nonparticipant_static_base["year_of_birth_final"].isna().sum())

print("\nMissing first_offense_date:")
print(nonparticipant_static_base["first_offense_date"].isna().sum())

print("\nMissing age_of_first_offense:")
print(nonparticipant_static_base["age_of_first_offense"].isna().sum())

print("\nGender distribution:")
print(nonparticipant_static_base["gender"].value_counts(dropna=False).head(10))

print("\nRace distribution:")
print(nonparticipant_static_base["race"].value_counts(dropna=False).head(10))

print("\nAge of first offense distribution:")
print(nonparticipant_static_base["age_of_first_offense"].describe())

print("\nTotal offense records distribution:")
print(nonparticipant_static_base["total_offense_records"].describe())

nonparticipant_static_base[[
    "pcs_off_id", "gender", "race", "year_of_birth_final",
    "first_offense_date", "age_of_first_offense", "total_offense_records"
]].head()


Missing DOB:
0

Missing year_of_birth_final:
0

Missing first_offense_date:
26

Missing age_of_first_offense:
26

Gender distribution:
gender
Male      93022
Female    30189
NaN         223
Name: count, dtype: int64

Race distribution:
race
White    78052
Black    42183
NaN       2579
Other      620
Name: count, dtype: int64

Age of first offense distribution:
count    123408.000000
mean         33.534803
std          11.685535
min         -24.000000
25%          24.000000
50%          31.000000
75%          41.000000
max          90.000000
Name: age_of_first_offense, dtype: float64

Total offense records distribution:
count    123433.000000
mean          2.816880
std           3.029715
min           0.000000
25%           1.000000
50%           2.000000
75%           3.000000
max          68.000000
Name: total_offense_records, dtype: float64


,pcs_off_id,gender,race,year_of_birth_final,first_offense_date,age_of_first_offense,total_offense_records
0,879948.0,Female,Other,1989.0,2018-07-27,28.0,3.0
1,885404.0,Female,White,1995.0,2019-02-01,23.0,1.0
2,1191002.0,Female,White,1951.0,NaT,NaN,0.0
3,1315519.0,Male,White,1975.0,2000-11-25,25.0,6.0
4,495537.0,Male,NaN,1989.0,2018-02-18,28.0,1.0


# Exact Matching

To construct the participant–non-participant comparison sample, we first apply exact matching on static characteristics that are defined prior to treatment and do not depend on the matched anchor date.

### Exact-Match Variables

We exact-match each TIP participant to non-participant candidates on:

- **gender**
- **race**
- **year of birth**
- **calendar age of first offense**

For participants:

- `gender` and `race` come from `all_tip_data`
- `year_of_birth` comes from the participant demographic fields
- `age_of_first_offense` is defined as the calendar age at the participant’s earliest observed offense date

For non-participants:

- `gender` and `race` come from `nonparticipant_static_base`
- `year_of_birth_final` is used as the final birth-year variable
- `age_of_first_offense` is defined as the calendar age at the person’s earliest observed offense date in PCS

### Purpose of Exact Matching

This step creates a candidate control pool that is tightly aligned with the participant on core demographic characteristics and baseline justice-system timing.

Only after this exact-match restriction do we assign the matched participant’s `StartDate`, `start_year`, and `cohort_2022` to the non-participant candidates and compute the dynamic pre-start variables used for propensity score matching.

### Matching Ratio

The exact-matching stage is designed to support **1:10 matching**, meaning that each participant may be matched to up to 10 non-participant candidates, subject to availability within the exact-match cell.

If fewer than 10 candidates are available in a given exact-match stratum, the participant remains matched to the available candidates only.

In [130]:
# --------------------------------------------------
# Block 1: Prepare participant exact-match base
# --------------------------------------------------

participant_exact_base = all_tip_data.copy()

# keep only rows with valid exact-match information
participant_exact_base["DOB"] = pd.to_datetime(participant_exact_base["DOB"], errors="coerce")
participant_exact_base["StartDate"] = pd.to_datetime(participant_exact_base["StartDate"], errors="coerce")

participant_exact_base["year_of_birth"] = pd.to_numeric(
    participant_exact_base["year_of_birth"], errors="coerce"
)

participant_exact_base["age_of_first_offense"] = pd.to_numeric(
    participant_exact_base["age_of_first_offense"], errors="coerce"
)

participant_exact_base["gender"] = participant_exact_base["gender"].fillna("Unknown").replace(0, "Unknown")
participant_exact_base["race"] = participant_exact_base["race"].fillna("Unknown").replace(0, "Unknown")

participant_exact_base = participant_exact_base[
    participant_exact_base["gender"].notna() &
    participant_exact_base["race"].notna() &
    participant_exact_base["year_of_birth"].notna() &
    (
        participant_exact_base["age_of_first_offense"].isna() |
        (participant_exact_base["age_of_first_offense"] >= 16)
    )
].copy()

print("participant_exact_base shape:", participant_exact_base.shape)
participant_exact_base[[
    "tip_id", "gender", "race", "year_of_birth", "age_of_first_offense",
    "StartDate", "cohort_2022"
]].head()

participant_exact_base shape: (1252, 230)


,tip_id,gender,race,year_of_birth,age_of_first_offense,StartDate,cohort_2022
2,3,Male,Black,1999.0,NaN,2018-07-23,0.0
3,4,Male,Black,1982.0,NaN,2013-08-12,0.0
4,5,Unknown,Unknown,0.0,NaN,2010-11-29,0.0
5,7,Male,Black,1990.0,19.0,2013-07-24,0.0
6,8,Male,Black,1989.0,20.0,2013-08-19,0.0


In [131]:
# --------------------------------------------------
# Block 2: Prepare non-participant exact-match base
# --------------------------------------------------

control_exact_base = nonparticipant_static_base.copy()

control_exact_base["year_of_birth_final"] = pd.to_numeric(
    control_exact_base["year_of_birth_final"], errors="coerce"
)

control_exact_base["age_of_first_offense"] = pd.to_numeric(
    control_exact_base["age_of_first_offense"], errors="coerce"
)

control_exact_base["gender"] = control_exact_base["gender"].fillna("Unknown").replace(0, "Unknown")
control_exact_base["race"] = control_exact_base["race"].fillna("Unknown").replace(0, "Unknown")

control_exact_base = control_exact_base[
    control_exact_base["gender"].notna() &
    control_exact_base["race"].notna() &
    control_exact_base["year_of_birth_final"].notna() &
    (
        control_exact_base["age_of_first_offense"].isna() |
        (control_exact_base["age_of_first_offense"] >= 16)
    )
].copy()

print("control_exact_base shape:", control_exact_base.shape)
control_exact_base[[
    "pcs_off_id", "gender", "race", "year_of_birth_final", "age_of_first_offense"
]].head()

control_exact_base shape: (123323, 25)


,pcs_off_id,gender,race,year_of_birth_final,age_of_first_offense
0,879948.0,Female,Other,1989.0,28.0
1,885404.0,Female,White,1995.0,23.0
2,1191002.0,Female,White,1951.0,NaN
3,1315519.0,Male,White,1975.0,25.0
4,495537.0,Male,Unknown,1989.0,28.0


In [132]:
# --------------------------------------------------
# Block 3: Quick diagnostics for exact-match variables
# --------------------------------------------------

print("\nParticipant gender distribution:")
print(participant_exact_base["gender"].value_counts(dropna=False).head(10))

print("\nParticipant race distribution:")
print(participant_exact_base["race"].value_counts(dropna=False).head(10))

print("\nParticipant year_of_birth summary:")
print(participant_exact_base["year_of_birth"].describe())

print("\nParticipant age_of_first_offense summary:")
print(participant_exact_base["age_of_first_offense"].describe())

print("\nControl gender distribution:")
print(control_exact_base["gender"].value_counts(dropna=False).head(10))

print("\nControl race distribution:")
print(control_exact_base["race"].value_counts(dropna=False).head(10))

print("\nControl year_of_birth_final summary:")
print(control_exact_base["year_of_birth_final"].describe())

print("\nControl age_of_first_offense summary:")
print(control_exact_base["age_of_first_offense"].describe())


Participant gender distribution:
gender
Male       905
Unknown    239
Female     108
Name: count, dtype: int64

Participant race distribution:
race
Black      901
Unknown    245
White      105
Other        1
Name: count, dtype: int64

Participant year_of_birth summary:
count    1252.000000
mean     1615.270767
std       778.847516
min         0.000000
25%      1976.000000
50%      1989.000000
75%      1996.000000
max      2006.000000
Name: year_of_birth, dtype: float64

Participant age_of_first_offense summary:
count    505.000000
mean      22.194059
std        5.642628
min       16.000000
25%       18.000000
50%       20.000000
75%       24.000000
max       53.000000
Name: age_of_first_offense, dtype: float64

Control gender distribution:
gender
Male       92913
Female     30187
Unknown      223
Name: count, dtype: int64

Control race distribution:
race
White      78012
Black      42114
Unknown     2577
Other        620
Name: count, dtype: int64

Control year_of_birth_final summary:


In [133]:
# --------------------------------------------------
# Block 4: Exact matching function
# --------------------------------------------------

def get_exact_match_candidates(participant_row, control_exact_base):
    """
    Return all non-participant candidates that exactly match one participant on:
    gender, race, year_of_birth, age_of_first_offense

    For age_of_first_offense:
    - exact numeric match if both are non-missing
    - treat NaN == NaN as a valid exact match
    """
    target_gender = participant_row["gender"]
    target_race = participant_row["race"]
    target_yob = participant_row["year_of_birth"]
    target_afo = participant_row["age_of_first_offense"]

    # age_of_first_offense exact match, with NaN-to-NaN matching allowed
    if pd.isna(target_afo):
        afo_match = control_exact_base["age_of_first_offense"].isna()
    else:
        afo_match = control_exact_base["age_of_first_offense"] == target_afo

    candidates = control_exact_base[
        (control_exact_base["gender"] == target_gender) &
        (control_exact_base["race"] == target_race) &
        (control_exact_base["year_of_birth_final"] == target_yob) &
        afo_match
    ].copy()

    return candidates

In [134]:
# --------------------------------------------------
# Block 5: Build exact-match candidate table
# --------------------------------------------------

exact_match_rows = []

for _, p_row in participant_exact_base.iterrows():
    candidates = get_exact_match_candidates(p_row, control_exact_base)

    if not candidates.empty:
        temp = candidates.copy()
        temp["tip_id"] = p_row["tip_id"]
        temp["participant_gender"] = p_row["gender"]
        temp["participant_race"] = p_row["race"]
        temp["participant_year_of_birth"] = p_row["year_of_birth"]
        temp["participant_age_of_first_offense"] = p_row["age_of_first_offense"]
        temp["participant_StartDate"] = p_row["StartDate"]
        temp["participant_cohort_2022"] = p_row["cohort_2022"]

        exact_match_rows.append(temp)

exact_match_candidates = (
    pd.concat(exact_match_rows, ignore_index=True)
    if exact_match_rows else pd.DataFrame()
)

print("exact_match_candidates shape:", exact_match_candidates.shape)
exact_match_candidates.head()

exact_match_candidates shape: (30571, 32)


,pcs_off_id,year_of_birth,year_of_death,census_tract,race,gender,off_race_from_pcs,off_sex_from_pcs,total_offense_records,total_sentencing_records,...,total_earnings_records,total_observed_earnings,ever_positive_earnings,tip_id,participant_gender,participant_race,participant_year_of_birth,participant_age_of_first_offense,participant_StartDate,participant_cohort_2022
0,1598117.0,1982.0,NaN,NaN,Black,Male,NaN,NaN,0.0,2.0,...,NaN,NaN,NaN,4,Male,Black,1982.0,NaN,2013-08-12,0.0
1,1576256.0,1990.0,NaN,4.200355e+10,Black,Male,NaN,NaN,11.0,11.0,...,31.0,0.000000,0.0,7,Male,Black,1990.0,19.0,2013-07-24,0.0
2,1176610.0,1990.0,NaN,4.200352e+10,Black,Male,NaN,NaN,5.0,5.0,...,31.0,0.000000,0.0,7,Male,Black,1990.0,19.0,2013-07-24,0.0
3,1327924.0,1990.0,NaN,4.200330e+10,Black,Male,NaN,NaN,12.0,12.0,...,33.0,40309.767409,1.0,7,Male,Black,1990.0,19.0,2013-07-24,0.0
4,1192323.0,1990.0,NaN,4.200326e+10,Black,Male,NaN,NaN,5.0,5.0,...,17.0,0.000000,0.0,7,Male,Black,1990.0,19.0,2013-07-24,0.0


In [135]:
# --------------------------------------------------
# Block 6: Count exact-match candidates per participant
# --------------------------------------------------

if not exact_match_candidates.empty:
    exact_match_counts = (
        exact_match_candidates.groupby("tip_id")
        .size()
        .reset_index(name="n_exact_candidates")
    )

    print(exact_match_counts["n_exact_candidates"].describe())

    print("\nShare of participants with at least 10 exact-match candidates:")
    print((exact_match_counts["n_exact_candidates"] >= 10).mean())

    print("\nDistribution of exact-match candidate counts:")
    print(exact_match_counts["n_exact_candidates"].value_counts().sort_index().head(20))
else:
    print("No exact-match candidates found.")

count    479.000000
mean      63.822547
std       42.563512
min        1.000000
25%       26.000000
50%       61.000000
75%       98.000000
max      206.000000
Name: n_exact_candidates, dtype: float64

Share of participants with at least 10 exact-match candidates:
0.8705636743215032

Distribution of exact-match candidate counts:
n_exact_candidates
1     33
2      7
3      2
4     10
5      1
6      5
7      1
8      2
9      1
10     8
11     4
12     2
13     4
14     7
15     3
16     5
17     2
18     1
19     3
20     2
Name: count, dtype: int64


In [136]:
# --------------------------------------------------
# Block 7: Temporary keep up to 10 exact-match candidates per participant
# --------------------------------------------------

if not exact_match_candidates.empty:
    exact_match_top10 = (
        exact_match_candidates
        .sort_values(["tip_id", "pcs_off_id"])
        .groupby("tip_id", group_keys=False)
        .head(10)
        .reset_index(drop=True)
    )

    print("exact_match_top10 shape:", exact_match_top10.shape)
    print("Unique participants in exact_match_top10:", exact_match_top10["tip_id"].nunique())
    exact_match_top10.head()
else:
    exact_match_top10 = pd.DataFrame()
    print("exact_match_top10 is empty.")

exact_match_top10 shape: (4330, 32)
Unique participants in exact_match_top10: 479


In [137]:
#rows_gr = []

#for _, p_row in participant_exact_base.iterrows():
    #cands = control_exact_base[
        #(control_exact_base["gender"] == p_row["gender"]) &
        #(control_exact_base["race"] == p_row["race"])
    #]
    #if not cands.empty:
        #rows_gr.append(p_row["tip_id"])

#print("Coverage with gender + race only:")
#print(len(set(rows_gr)) / participant_exact_base["tip_id"].nunique())

In [138]:
#rows_gry = []

#for _, p_row in participant_exact_base.iterrows():
    #cands = control_exact_base[
        #(control_exact_base["gender"] == p_row["gender"]) &
        #(control_exact_base["race"] == p_row["race"]) &
        #(control_exact_base["year_of_birth_final"] == p_row["year_of_birth"])
    #]
    #if not cands.empty:
        #rows_gry.append(p_row["tip_id"])

#print("Coverage with gender + race + year_of_birth:")
#print(len(set(rows_gry)) / participant_exact_base["tip_id"].nunique())

# Dynamic Pre-Start Covariates for Exact-Matched Non-Participant Candidates

After exact matching on gender, race, year of birth, and calendar age of first offense, each non-participant candidate inherits the matched participant’s treatment timing variables:

- **StartDate**
- **start year**
- **cohort_2022**

These inherited values define the candidate’s comparison baseline and ensure that pre-treatment covariates are measured relative to the same time anchor as the matched participant.

### Inherited Timing Variables

For each exact-matched participant–candidate pair:

- `inherited_StartDate` is set equal to the matched participant’s TIP `StartDate`
- `start_year` is the calendar year of `inherited_StartDate`
- `cohort_2022` is inherited from the matched participant and indicates whether the anchor date falls before or after January 1, 2022

### Dynamic Pre-Start Covariates

Using the inherited `StartDate`, we construct the following non-participant baseline covariates:

- **num_sentences_before_start**: number of sentencing records with `DOS < inherited_StartDate`
- **any_high_ogs_before_start**: indicator for whether any pre-start sentencing record has `OGS > 5`
- **num_offenses_before_start**: number of offense records with `DOF < inherited_StartDate`
- **earnings_2yr_preTIP**: total adjusted earnings in the second year before the inherited start date
- **job_right_before_tip**: indicator for whether the person has any positive earnings in the year immediately before the inherited start date

These variables are constructed separately for each participant–candidate pair, because the same non-participant may appear in multiple exact-match cells and therefore inherit different start dates from different participants.

In [139]:
# --------------------------------------------------
# Block 1: Prepare exact-match pair base with inherited timing
# --------------------------------------------------

exact_match_pairs = exact_match_candidates.copy()

exact_match_pairs["participant_StartDate"] = pd.to_datetime(
    exact_match_pairs["participant_StartDate"], errors="coerce"
)

exact_match_pairs["inherited_StartDate"] = exact_match_pairs["participant_StartDate"]
exact_match_pairs["start_year"] = exact_match_pairs["inherited_StartDate"].dt.year
exact_match_pairs["cohort_2022"] = exact_match_pairs["participant_cohort_2022"]

print("exact_match_pairs shape:", exact_match_pairs.shape)
exact_match_pairs[[
    "tip_id", "pcs_off_id", "inherited_StartDate", "start_year", "cohort_2022"
]].head()

exact_match_pairs shape: (30571, 35)


,tip_id,pcs_off_id,inherited_StartDate,start_year,cohort_2022
0,4,1598117.0,2013-08-12,2013,0.0
1,7,1576256.0,2013-07-24,2013,0.0
2,7,1176610.0,2013-07-24,2013,0.0
3,7,1327924.0,2013-07-24,2013,0.0
4,7,1192323.0,2013-07-24,2013,0.0


In [140]:
# --------------------------------------------------
# Block 2: Dynamic sentencing features for each exact-match pair
# --------------------------------------------------

pair_sent = exact_match_pairs[[
    "tip_id", "pcs_off_id", "inherited_StartDate"
]].drop_duplicates().copy()

pcs_sentencing = nonparticipant_pcs_long[
    nonparticipant_pcs_long["DOS"].notna()
].copy()

pair_sent_merged = pair_sent.merge(
    pcs_sentencing[["pcs_off_id", "DOS", "OGS"]],
    on="pcs_off_id",
    how="left"
)

pair_sent_before = pair_sent_merged[
    pair_sent_merged["DOS"].notna() &
    pair_sent_merged["inherited_StartDate"].notna() &
    (pair_sent_merged["DOS"] < pair_sent_merged["inherited_StartDate"])
].copy()

pair_sent_features = (
    pair_sent_before.groupby(["tip_id", "pcs_off_id", "inherited_StartDate"], as_index=False)
    .agg(
        num_sentences_before_start=("DOS", "count"),
        any_high_ogs_before_start=("OGS", lambda x: 1 if (x > 5).any() else 0)
    )
)

print("pair_sent_features shape:", pair_sent_features.shape)
pair_sent_features.head()

pair_sent_features shape: (27439, 5)


,tip_id,pcs_off_id,inherited_StartDate,num_sentences_before_start,any_high_ogs_before_start
0,4,1598117.0,2013-08-12,2,0
1,7,1014531.0,2013-07-24,2,1
2,7,1020443.0,2013-07-24,4,1
3,7,1026120.0,2013-07-24,3,1
4,7,1040347.0,2013-07-24,7,1


In [141]:
# --------------------------------------------------
# Block 3: Dynamic offense features for each exact-match pair
# --------------------------------------------------

pair_off = exact_match_pairs[[
    "tip_id", "pcs_off_id", "inherited_StartDate"
]].drop_duplicates().copy()

pcs_offense = nonparticipant_pcs_long[
    nonparticipant_pcs_long["DOF"].notna()
].copy()

pair_off_merged = pair_off.merge(
    pcs_offense[["pcs_off_id", "DOF"]],
    on="pcs_off_id",
    how="left"
)

pair_off_before = pair_off_merged[
    pair_off_merged["DOF"].notna() &
    pair_off_merged["inherited_StartDate"].notna() &
    (pair_off_merged["DOF"] < pair_off_merged["inherited_StartDate"])
].copy()

pair_off_features = (
    pair_off_before.groupby(["tip_id", "pcs_off_id", "inherited_StartDate"], as_index=False)
    .agg(
        num_offenses_before_start=("DOF", "count")
    )
)

print("pair_off_features shape:", pair_off_features.shape)
pair_off_features.head()

pair_off_features shape: (29478, 4)


,tip_id,pcs_off_id,inherited_StartDate,num_offenses_before_start
0,7,1014531.0,2013-07-24,2
1,7,1020443.0,2013-07-24,4
2,7,1026120.0,2013-07-24,3
3,7,1040347.0,2013-07-24,7
4,7,1046210.0,2013-07-24,6


In [142]:
# --------------------------------------------------
# Block 4: Dynamic earnings features for each exact-match pair
# --------------------------------------------------

pair_earn = exact_match_pairs[[
    "tip_id", "pcs_off_id", "inherited_StartDate"
]].drop_duplicates().copy()

pair_earn_merged = pair_earn.merge(
    nonparticipant_earnings_long[["pcs_off_id", "year_quarter_dt", "adjusted_earnings"]],
    on="pcs_off_id",
    how="left"
)

# window for earnings_2yr_preTIP: second year before start
pair_earn_merged["pre2_lower"] = pair_earn_merged["inherited_StartDate"] - pd.Timedelta(days=365 * 2)
pair_earn_merged["pre1_lower"] = pair_earn_merged["inherited_StartDate"] - pd.Timedelta(days=365)

earn_pre2 = pair_earn_merged[
    pair_earn_merged["year_quarter_dt"].notna() &
    pair_earn_merged["inherited_StartDate"].notna() &
    (pair_earn_merged["year_quarter_dt"] >= pair_earn_merged["pre2_lower"]) &
    (pair_earn_merged["year_quarter_dt"] < pair_earn_merged["pre1_lower"])
].copy()

pair_earn2_features = (
    earn_pre2.groupby(["tip_id", "pcs_off_id", "inherited_StartDate"], as_index=False)
    .agg(
        earnings_2yr_preTIP=("adjusted_earnings", "sum")
    )
)

# window for job_right_before_tip: immediate 1 year before start
earn_pre1 = pair_earn_merged[
    pair_earn_merged["year_quarter_dt"].notna() &
    pair_earn_merged["inherited_StartDate"].notna() &
    (pair_earn_merged["year_quarter_dt"] >= pair_earn_merged["pre1_lower"]) &
    (pair_earn_merged["year_quarter_dt"] < pair_earn_merged["inherited_StartDate"])
].copy()

pair_job_features = (
    earn_pre1.groupby(["tip_id", "pcs_off_id", "inherited_StartDate"], as_index=False)
    .agg(
        job_right_before_tip=("adjusted_earnings", lambda x: 1 if (x > 0).any() else 0)
    )
)

print("pair_earn2_features shape:", pair_earn2_features.shape)
print("pair_job_features shape:", pair_job_features.shape)

pair_earn2_features shape: (12896, 4)
pair_job_features shape: (16445, 4)


In [143]:
# --------------------------------------------------
# Block 5: Merge dynamic features back to pair table
# --------------------------------------------------

exact_match_pairs_dynamic = exact_match_pairs.merge(
    pair_sent_features,
    on=["tip_id", "pcs_off_id", "inherited_StartDate"],
    how="left"
)

exact_match_pairs_dynamic = exact_match_pairs_dynamic.merge(
    pair_off_features,
    on=["tip_id", "pcs_off_id", "inherited_StartDate"],
    how="left"
)

exact_match_pairs_dynamic = exact_match_pairs_dynamic.merge(
    pair_earn2_features,
    on=["tip_id", "pcs_off_id", "inherited_StartDate"],
    how="left"
)

exact_match_pairs_dynamic = exact_match_pairs_dynamic.merge(
    pair_job_features,
    on=["tip_id", "pcs_off_id", "inherited_StartDate"],
    how="left"
)

fill_zero_cols = [
    "num_sentences_before_start",
    "any_high_ogs_before_start",
    "num_offenses_before_start",
    "earnings_2yr_preTIP",
    "job_right_before_tip"
]

for col in fill_zero_cols:
    if col in exact_match_pairs_dynamic.columns:
        exact_match_pairs_dynamic[col] = exact_match_pairs_dynamic[col].fillna(0)

print("exact_match_pairs_dynamic shape:", exact_match_pairs_dynamic.shape)
exact_match_pairs_dynamic[[
    "tip_id", "pcs_off_id", "inherited_StartDate", "start_year", "cohort_2022",
    "num_sentences_before_start", "any_high_ogs_before_start",
    "num_offenses_before_start", "earnings_2yr_preTIP", "job_right_before_tip"
]].head()

exact_match_pairs_dynamic shape: (30571, 40)


,tip_id,pcs_off_id,inherited_StartDate,start_year,cohort_2022,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip
0,4,1598117.0,2013-08-12,2013,0.0,2.0,0.0,0.0,0.0,0.0
1,7,1576256.0,2013-07-24,2013,0.0,6.0,1.0,6.0,0.0,0.0
2,7,1176610.0,2013-07-24,2013,0.0,2.0,0.0,2.0,0.0,0.0
3,7,1327924.0,2013-07-24,2013,0.0,10.0,1.0,10.0,0.0,0.0
4,7,1192323.0,2013-07-24,2013,0.0,5.0,1.0,5.0,0.0,0.0


In [144]:
# --------------------------------------------------
# Block 6: Quick checks
# --------------------------------------------------

dynamic_vars = [
    "num_sentences_before_start",
    "any_high_ogs_before_start",
    "num_offenses_before_start",
    "earnings_2yr_preTIP",
    "job_right_before_tip"
]

print("\nSummary of dynamic variables:")
print(exact_match_pairs_dynamic[dynamic_vars].describe())

print("\nShare of candidates with positive earnings_2yr_preTIP:")
print((exact_match_pairs_dynamic["earnings_2yr_preTIP"] > 0).mean())

print("\nShare of candidates with job_right_before_tip = 1:")
print(exact_match_pairs_dynamic["job_right_before_tip"].mean())

print("\nShare of candidates with any_high_ogs_before_start = 1:")
print(exact_match_pairs_dynamic["any_high_ogs_before_start"].mean())


Summary of dynamic variables:
       num_sentences_before_start  any_high_ogs_before_start  \
count                30571.000000               30571.000000   
mean                     3.586471                   0.510778   
std                      3.559027                   0.499892   
min                      0.000000                   0.000000   
25%                      1.000000                   0.000000   
50%                      2.000000                   1.000000   
75%                      5.000000                   1.000000   
max                     70.000000                   1.000000   

       num_offenses_before_start  earnings_2yr_preTIP  job_right_before_tip  
count               30571.000000         30571.000000          30571.000000  
mean                    3.897714          2866.126669              0.257106  
std                     3.621579          9836.403577              0.437046  
min                     0.000000             0.000000              0.000000  
25

# Within-Cell Matching on Dynamic Pre-Start Covariates

After exact matching on gender, race, year of birth, and calendar age of first offense, we further refine the control pool using dynamic pre-start covariates measured relative to the matched participant’s inherited `StartDate`.

### Dynamic Covariates Used for Matching

Within each exact-match cell, we compare the participant and non-participant candidates on the following baseline variables:

- **num_sentences_before_start**
- **any_high_ogs_before_start**
- **num_offenses_before_start**
- **earnings_2yr_preTIP**
- **job_right_before_tip**

For participants, these variables are measured before observed TIP `StartDate`.  
For non-participants, the same variables are measured before the inherited `StartDate` from the matched participant.

### Matching Procedure

Within each exact-match cell, we standardize the five baseline covariates and compute the distance between the participant and each non-participant candidate.

We then select the **10 nearest non-participants** for each participant.

This procedure serves the same purpose as propensity score refinement after exact matching: it retains control observations that are most similar to the participant on pre-treatment criminal-history and earnings characteristics, while avoiding instability from estimating a separate binary propensity score model in very small exact-match cells.

### Output

The result is a participant–control matched-pair table in long format, where each participant may contribute up to 10 matched non-participants.

In [145]:
# --------------------------------------------------
# Block 1: Prepare participant-side matching variables
# --------------------------------------------------

participant_psm_base = all_tip_data.copy()

participant_psm_vars = [
    "tip_id",
    "StartDate",
    "cohort_2022",
    "gender",
    "race",
    "year_of_birth",
    "age_of_first_offense",
    "num_sentences_before_start",
    "any_high_ogs_before_start",
    "num_offenses_before_start",
    "earnings_2yr_preTIP",
    "job_right_before_tip"
]

participant_psm_base = participant_psm_base[participant_psm_vars].copy()

# clean types
participant_psm_base["StartDate"] = pd.to_datetime(participant_psm_base["StartDate"], errors="coerce")
participant_psm_base["year_of_birth"] = pd.to_numeric(participant_psm_base["year_of_birth"], errors="coerce")
participant_psm_base["age_of_first_offense"] = pd.to_numeric(participant_psm_base["age_of_first_offense"], errors="coerce")

for col in [
    "num_sentences_before_start",
    "any_high_ogs_before_start",
    "num_offenses_before_start",
    "earnings_2yr_preTIP",
    "job_right_before_tip"
]:
    participant_psm_base[col] = pd.to_numeric(participant_psm_base[col], errors="coerce").fillna(0)

print("participant_psm_base shape:", participant_psm_base.shape)
participant_psm_base.head()

participant_psm_base shape: (1415, 12)


,tip_id,StartDate,cohort_2022,gender,race,year_of_birth,age_of_first_offense,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip
0,1,2018-07-12,0.0,Male,Black,1992.0,9.0,3.0,1.0,2.0,410.102835,0.0
1,2,2018-07-12,0.0,Male,Black,1997.0,13.0,2.0,1.0,2.0,8049.772420,1.0
2,3,2018-07-23,0.0,Male,Black,1999.0,NaN,0.0,0.0,0.0,0.000000,0.0
3,4,2013-08-12,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.000000,0.0
4,5,2010-11-29,0.0,0,0,0.0,NaN,0.0,0.0,0.0,0.000000,0.0


In [146]:
# --------------------------------------------------
# Block 2: Merge participant PSM variables into pair table
# --------------------------------------------------

exact_match_pairs_dynamic = exact_match_pairs_dynamic.merge(
    participant_psm_base[[
        "tip_id",
        "num_sentences_before_start",
        "any_high_ogs_before_start",
        "num_offenses_before_start",
        "earnings_2yr_preTIP",
        "job_right_before_tip"
    ]].rename(columns={
        "num_sentences_before_start": "participant_num_sentences_before_start",
        "any_high_ogs_before_start": "participant_any_high_ogs_before_start",
        "num_offenses_before_start": "participant_num_offenses_before_start",
        "earnings_2yr_preTIP": "participant_earnings_2yr_preTIP",
        "job_right_before_tip": "participant_job_right_before_tip"
    }),
    on="tip_id",
    how="left"
)

print("exact_match_pairs_dynamic shape after participant merge:", exact_match_pairs_dynamic.shape)
exact_match_pairs_dynamic.head()

exact_match_pairs_dynamic shape after participant merge: (30571, 45)


,pcs_off_id,year_of_birth,year_of_death,census_tract,race,gender,off_race_from_pcs,off_sex_from_pcs,total_offense_records,total_sentencing_records,...,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip,participant_num_sentences_before_start,participant_any_high_ogs_before_start,participant_num_offenses_before_start,participant_earnings_2yr_preTIP,participant_job_right_before_tip
0,1598117.0,1982.0,NaN,NaN,Black,Male,NaN,NaN,0.0,2.0,...,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1576256.0,1990.0,NaN,4.200355e+10,Black,Male,NaN,NaN,11.0,11.0,...,6.0,1.0,6.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
2,1176610.0,1990.0,NaN,4.200352e+10,Black,Male,NaN,NaN,5.0,5.0,...,2.0,0.0,2.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
3,1327924.0,1990.0,NaN,4.200330e+10,Black,Male,NaN,NaN,12.0,12.0,...,10.0,1.0,10.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
4,1192323.0,1990.0,NaN,4.200326e+10,Black,Male,NaN,NaN,5.0,5.0,...,5.0,1.0,5.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0


In [147]:
# --------------------------------------------------
# Block 3: Compute within-cell standardized distance
# --------------------------------------------------

match_vars_control = [
    "num_sentences_before_start",
    "any_high_ogs_before_start",
    "num_offenses_before_start",
    "earnings_2yr_preTIP",
    "job_right_before_tip"
]

match_vars_participant = [
    "participant_num_sentences_before_start",
    "participant_any_high_ogs_before_start",
    "participant_num_offenses_before_start",
    "participant_earnings_2yr_preTIP",
    "participant_job_right_before_tip"
]

var_map = dict(zip(match_vars_control, match_vars_participant))


def compute_within_tip_distance(df_group):
    """
    df_group contains one participant's exact-match candidate pool.
    Compute standardized Euclidean distance between the participant and each candidate.
    """
    df_group = df_group.copy()

    # means and stds from control candidates in this exact-match cell
    means = df_group[match_vars_control].mean()
    stds = df_group[match_vars_control].std()

    # avoid division by zero
    stds = stds.replace(0, 1).fillna(1)

    total_distance = np.zeros(len(df_group))

    for c_var, p_var in var_map.items():
        control_z = (df_group[c_var] - means[c_var]) / stds[c_var]
        participant_value = df_group[p_var].iloc[0]
        participant_z = (participant_value - means[c_var]) / stds[c_var]

        total_distance += (control_z - participant_z) ** 2

    df_group["psm_distance"] = np.sqrt(total_distance)
    return df_group


exact_match_pairs_scored = (
    exact_match_pairs_dynamic
    .groupby("tip_id", group_keys=False)
    .apply(compute_within_tip_distance)
    .reset_index(drop=True)
)

print("exact_match_pairs_scored shape:", exact_match_pairs_scored.shape)
exact_match_pairs_scored[[
    "tip_id", "pcs_off_id", "psm_distance"
]].head()

exact_match_pairs_scored shape: (30571, 46)


C:\Users\13429\AppData\Local\Temp\ipykernel_33308\658760534.py:54: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_within_tip_distance)


,tip_id,pcs_off_id,psm_distance
0,4,1598117.0,2.000000
1,7,1576256.0,2.410722
2,7,1176610.0,2.065505
3,7,1327924.0,4.339299
4,7,1192323.0,1.928577


In [148]:
# --------------------------------------------------
# Block 4: Keep top 10 matched controls per participant
# --------------------------------------------------

matched_top1_controls = (
    exact_match_pairs_scored
    .sort_values(["tip_id", "psm_distance", "pcs_off_id"])
    .groupby("tip_id", group_keys=False)
    .head(1)
    .reset_index(drop=True)
)

print("matched_top1_controls shape:", matched_top1_controls.shape)
print("Unique matched participants:", matched_top1_controls["tip_id"].nunique())

matched_top1_controls[[
    "tip_id", "pcs_off_id", "psm_distance",
    "num_sentences_before_start", "any_high_ogs_before_start",
    "num_offenses_before_start", "earnings_2yr_preTIP", "job_right_before_tip"
]].head(20)

matched_top1_controls shape: (479, 46)
Unique matched participants: 479


,tip_id,pcs_off_id,psm_distance,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip
0,4,1598117.0,2.000000,2.0,0.0,0.0,0.000000,0.0
1,7,1248203.0,0.000000,1.0,1.0,1.0,0.000000,0.0
2,8,1018143.0,0.000000,1.0,0.0,2.0,0.000000,0.0
3,10,1021892.0,0.380005,2.0,1.0,2.0,0.000000,0.0
4,12,1365128.0,1.036302,2.0,1.0,2.0,0.000000,0.0
5,16,1619463.0,0.676698,2.0,1.0,3.0,0.000000,0.0
6,17,1059572.0,0.000000,0.0,0.0,1.0,0.000000,0.0
7,18,1209944.0,2.763863,5.0,0.0,5.0,0.000000,0.0
8,19,1120626.0,0.512270,3.0,1.0,3.0,0.000000,0.0
9,20,1045347.0,0.000000,1.0,0.0,2.0,0.000000,0.0


In [149]:
# --------------------------------------------------
# Block 5: Diagnostics for 1:10 matching
# --------------------------------------------------

matched_counts = (
    matched_top1_controls.groupby("tip_id")
    .size()
    .reset_index(name="n_matched_controls")
)

print("\nDistribution of matched control counts:")
print(matched_counts["n_matched_controls"].value_counts().sort_index())

print("\nShare of participants with 10 matched controls:")
print((matched_counts["n_matched_controls"] == 10).mean())

print("\nPSM distance summary:")
print(matched_top1_controls["psm_distance"].describe())


Distribution of matched control counts:
n_matched_controls
1    479
Name: count, dtype: int64

Share of participants with 10 matched controls:
0.0

PSM distance summary:
count      479.000000
mean       475.055528
std       3507.929567
min          0.000000
25%          0.310864
50%          0.625642
75%          1.599201
max      47871.170868
Name: psm_distance, dtype: float64


In [150]:
# --------------------------------------------------
# Block 6: Build final matched long table
# --------------------------------------------------

# control rows
matched_control_long = matched_top1_controls.copy()
matched_control_long["unit_type"] = "control"
matched_control_long["treatment"] = 0

matched_control_long = matched_control_long.rename(columns={
    "inherited_StartDate": "StartDate"
})

# keep harmonized control-side columns
matched_control_long = matched_control_long[[
    "tip_id", "pcs_off_id", "unit_type", "treatment",
    "StartDate", "start_year", "cohort_2022",
    "gender", "race", "year_of_birth_final", "age_of_first_offense",
    "num_sentences_before_start", "any_high_ogs_before_start",
    "num_offenses_before_start", "earnings_2yr_preTIP",
    "job_right_before_tip", "psm_distance"
]].copy()

matched_control_long = matched_control_long.rename(columns={
    "year_of_birth_final": "year_of_birth"
})

# participant rows
matched_tip_ids = matched_top1_controls["tip_id"].unique()

matched_participant_long = participant_psm_base[
    participant_psm_base["tip_id"].isin(matched_tip_ids)
].copy()

matched_participant_long["pcs_off_id"] = np.nan
matched_participant_long["unit_type"] = "participant"
matched_participant_long["treatment"] = 1
matched_participant_long["start_year"] = matched_participant_long["StartDate"].dt.year
matched_participant_long["psm_distance"] = 0.0

matched_participant_long = matched_participant_long[[
    "tip_id", "pcs_off_id", "unit_type", "treatment",
    "StartDate", "start_year", "cohort_2022",
    "gender", "race", "year_of_birth", "age_of_first_offense",
    "num_sentences_before_start", "any_high_ogs_before_start",
    "num_offenses_before_start", "earnings_2yr_preTIP",
    "job_right_before_tip", "psm_distance"
]].copy()

# combine
matched_psm_long = pd.concat(
    [matched_participant_long, matched_control_long],
    ignore_index=True
)

print("matched_psm_long shape:", matched_psm_long.shape)
print("Participant rows:", (matched_psm_long["treatment"] == 1).sum())
print("Control rows:", (matched_psm_long["treatment"] == 0).sum())

matched_psm_long.head()

matched_psm_long shape: (958, 17)
Participant rows: 479
Control rows: 479


,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,year_of_birth,age_of_first_offense,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip,psm_distance
0,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
1,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,1990.0,19.0,1.0,1.0,1.0,0.0,0.0,0.0
2,8,NaN,participant,1,2013-08-19,2013,0.0,Male,Black,1989.0,20.0,1.0,0.0,2.0,0.0,0.0,0.0
3,10,NaN,participant,1,2010-07-09,2010,0.0,Male,Black,1989.0,18.0,2.0,1.0,1.0,0.0,0.0,0.0
4,12,NaN,participant,1,2013-02-27,2013,0.0,Male,Black,1987.0,23.0,3.0,1.0,1.0,0.0,0.0,0.0


In [151]:
# --------------------------------------------------
# Block 7: Quick balance check on matching variables
# --------------------------------------------------

balance_vars = [
    "num_sentences_before_start",
    "any_high_ogs_before_start",
    "num_offenses_before_start",
    "earnings_2yr_preTIP",
    "job_right_before_tip"
]

print(
    matched_psm_long.groupby("treatment")[balance_vars]
    .mean()
)

           num_sentences_before_start  any_high_ogs_before_start  \
treatment                                                          
0                            3.411273                   0.511482   
1                            4.135699                   0.494781   

           num_offenses_before_start  earnings_2yr_preTIP  \
treatment                                                   
0                           3.532359          3039.026512   
1                           2.835073          3757.859328   

           job_right_before_tip  
treatment                        
0                      0.340292  
1                      0.390397  


# Defining Outcomes for the 1:10 Matched Participant–Non-Participant Sample

After exact matching and within-cell matching on dynamic pre-start covariates, we define post-start outcomes for both participants and matched non-participants.

### Outcome Time Window

For each observation, outcomes are measured relative to that row’s `StartDate`:

- for TIP participants, `StartDate` is the observed TIP start date
- for matched non-participants, `StartDate` is the inherited anchor date from the matched participant

All outcomes are defined within **1.5 years after StartDate**.

### Outcomes

We define the following outcomes:

- **employment_outcome**: indicator for whether the person has any positive earnings within 1.5 years after StartDate
- **earnings_outcome**: total adjusted earnings within 1.5 years after StartDate
- **offense_outcome**: indicator for whether the person has any offense record within 1.5 years after StartDate
- **recidivism_outcome**: indicator for whether the person has at least one offense before StartDate and at least one offense within 1.5 years after StartDate

### Data Sources

For participants:

- earnings outcomes are constructed from `tip_earnings`
- offense and recidivism outcomes are constructed from the combined participant offense-date table, which includes both sentencing `DOF` records and additional `OffenseDate` records from `tip_arrests` when not already present in sentencing

For non-participants:

- earnings outcomes are constructed from `nonparticipant_earnings_long`
- offense and recidivism outcomes are constructed from `nonparticipant_pcs_long`

This design ensures that both groups are evaluated over the same follow-up window and that recidivism is defined symmetrically as a pre-start offense history followed by a post-start offense event.

In [152]:
# --------------------------------------------------
# Block 1: Start from matched sample
# --------------------------------------------------

matched_outcome_df = matched_psm_long.copy()

matched_outcome_df["StartDate"] = pd.to_datetime(matched_outcome_df["StartDate"], errors="coerce")

print("matched_outcome_df shape:", matched_outcome_df.shape)
matched_outcome_df.head()

matched_outcome_df shape: (958, 17)


,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,year_of_birth,age_of_first_offense,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip,psm_distance
0,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
1,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,1990.0,19.0,1.0,1.0,1.0,0.0,0.0,0.0
2,8,NaN,participant,1,2013-08-19,2013,0.0,Male,Black,1989.0,20.0,1.0,0.0,2.0,0.0,0.0,0.0
3,10,NaN,participant,1,2010-07-09,2010,0.0,Male,Black,1989.0,18.0,2.0,1.0,1.0,0.0,0.0,0.0
4,12,NaN,participant,1,2013-02-27,2013,0.0,Male,Black,1987.0,23.0,3.0,1.0,1.0,0.0,0.0,0.0


In [153]:
# --------------------------------------------------
# Block 2: Participant earnings outcomes
# --------------------------------------------------

participant_rows = matched_outcome_df[
    matched_outcome_df["treatment"] == 1
][["tip_id", "StartDate"]].drop_duplicates().copy()

participant_earn = participant_rows.merge(
    tip_earnings[["tip_id", "year_quarter_dt", "adjusted_earnings"]],
    on="tip_id",
    how="left"
)

participant_earn["upper_date"] = participant_earn["StartDate"] + pd.Timedelta(days=int(365.25 * 1.5))

participant_earn_post = participant_earn[
    participant_earn["year_quarter_dt"].notna() &
    participant_earn["StartDate"].notna() &
    (participant_earn["year_quarter_dt"] > participant_earn["StartDate"]) &
    (participant_earn["year_quarter_dt"] < participant_earn["upper_date"])
].copy()

participant_earn_outcomes = (
    participant_earn_post.groupby(["tip_id", "StartDate"], as_index=False)
    .agg(
        employment_outcome=("adjusted_earnings", lambda x: 1 if (x != 0).any() else 0),
        earnings_outcome=("adjusted_earnings", "sum")
    )
)

participant_earn_outcomes.head()

,tip_id,StartDate,employment_outcome,earnings_outcome
0,210,2020-07-27,0,0.000000
1,341,2017-06-12,1,218.020030
2,342,2015-10-28,1,1722.609768
3,348,2015-11-09,0,0.000000
4,359,2015-12-07,1,1173.660822


In [154]:
# --------------------------------------------------
# Block 3: Participant offense and recidivism outcomes
# --------------------------------------------------

participant_off = participant_rows.merge(
    combined_offense_dates[["tip_id", "offense_date"]],
    on="tip_id",
    how="left"
)

participant_off["upper_date"] = participant_off["StartDate"] + pd.Timedelta(days=int(365.25 * 1.5))

participant_off_prior = participant_off[
    participant_off["offense_date"].notna() &
    participant_off["StartDate"].notna() &
    (participant_off["offense_date"] < participant_off["StartDate"])
].copy()

participant_off_post = participant_off[
    participant_off["offense_date"].notna() &
    participant_off["StartDate"].notna() &
    (participant_off["offense_date"] > participant_off["StartDate"]) &
    (participant_off["offense_date"] < participant_off["upper_date"])
].copy()

participant_prior_summary = (
    participant_off_prior.groupby(["tip_id", "StartDate"], as_index=False)
    .agg(
        prior_offense_before_start=("offense_date", lambda x: 1 if x.count() > 0 else 0)
    )
)

participant_post_summary = (
    participant_off_post.groupby(["tip_id", "StartDate"], as_index=False)
    .agg(
        offense_outcome=("offense_date", lambda x: 1 if x.count() > 0 else 0)
    )
)

participant_off_outcomes = participant_rows.merge(
    participant_prior_summary,
    on=["tip_id", "StartDate"],
    how="left"
).merge(
    participant_post_summary,
    on=["tip_id", "StartDate"],
    how="left"
)

participant_off_outcomes["prior_offense_before_start"] = (
    participant_off_outcomes["prior_offense_before_start"].fillna(0)
)

participant_off_outcomes["offense_outcome"] = (
    participant_off_outcomes["offense_outcome"].fillna(0)
)

participant_off_outcomes["recidivism_outcome"] = np.where(
    (participant_off_outcomes["prior_offense_before_start"] > 0) &
    (participant_off_outcomes["offense_outcome"] > 0),
    1,
    0
)

participant_off_outcomes.head()

,tip_id,StartDate,prior_offense_before_start,offense_outcome,recidivism_outcome
0,4,2013-08-12,0.0,0.0,0
1,7,2013-07-24,1.0,1.0,1
2,8,2013-08-19,1.0,0.0,0
3,10,2010-07-09,1.0,0.0,0
4,12,2013-02-27,1.0,0.0,0


In [155]:
# --------------------------------------------------
# Block 4: Merge participant outcomes back
# --------------------------------------------------

participant_outcomes = participant_rows.merge(
    participant_earn_outcomes,
    on=["tip_id", "StartDate"],
    how="left"
).merge(
    participant_off_outcomes[[
        "tip_id", "StartDate", "offense_outcome", "recidivism_outcome"
    ]],
    on=["tip_id", "StartDate"],
    how="left"
)

for col in ["employment_outcome", "earnings_outcome", "offense_outcome", "recidivism_outcome"]:
    if col in participant_outcomes.columns:
        participant_outcomes[col] = participant_outcomes[col].fillna(0)

participant_outcomes.head()

,tip_id,StartDate,employment_outcome,earnings_outcome,offense_outcome,recidivism_outcome
0,4,2013-08-12,0.0,0.0,0.0,0
1,7,2013-07-24,0.0,0.0,1.0,1
2,8,2013-08-19,0.0,0.0,0.0,0
3,10,2010-07-09,0.0,0.0,0.0,0
4,12,2013-02-27,0.0,0.0,0.0,0


In [156]:
# --------------------------------------------------
# Block 5: Control earnings outcomes
# --------------------------------------------------

control_rows = matched_outcome_df[
    matched_outcome_df["treatment"] == 0
][["tip_id", "pcs_off_id", "StartDate"]].drop_duplicates().copy()

control_earn = control_rows.merge(
    nonparticipant_earnings_long[["pcs_off_id", "year_quarter_dt", "adjusted_earnings"]],
    on="pcs_off_id",
    how="left"
)

control_earn["upper_date"] = control_earn["StartDate"] + pd.Timedelta(days=int(365.25 * 1.5))

control_earn_post = control_earn[
    control_earn["year_quarter_dt"].notna() &
    control_earn["StartDate"].notna() &
    (control_earn["year_quarter_dt"] > control_earn["StartDate"]) &
    (control_earn["year_quarter_dt"] < control_earn["upper_date"])
].copy()

control_earn_outcomes = (
    control_earn_post.groupby(["tip_id", "pcs_off_id", "StartDate"], as_index=False)
    .agg(
        employment_outcome=("adjusted_earnings", lambda x: 1 if (x > 0).any() else 0),
        earnings_outcome=("adjusted_earnings", "sum")
    )
)

control_earn_outcomes.head()

,tip_id,pcs_off_id,StartDate,employment_outcome,earnings_outcome
0,210,1498813.0,2020-07-27,1,28147.814532
1,341,1149473.0,2017-06-12,1,699.612302
2,342,1085158.0,2015-10-28,0,0.000000
3,359,1040548.0,2015-12-07,0,0.000000
4,360,1228960.0,2015-12-14,1,6220.535273


In [157]:
# --------------------------------------------------
# Block 6: Control offense and recidivism outcomes
# --------------------------------------------------

control_off = control_rows.merge(
    nonparticipant_pcs_long[["pcs_off_id", "DOF"]],
    on="pcs_off_id",
    how="left"
)

control_off = control_off.rename(columns={"DOF": "offense_date"})

control_off["upper_date"] = control_off["StartDate"] + pd.Timedelta(days=int(365.25 * 1.5))

control_off_prior = control_off[
    control_off["offense_date"].notna() &
    control_off["StartDate"].notna() &
    (control_off["offense_date"] < control_off["StartDate"])
].copy()

control_off_post = control_off[
    control_off["offense_date"].notna() &
    control_off["StartDate"].notna() &
    (control_off["offense_date"] > control_off["StartDate"]) &
    (control_off["offense_date"] < control_off["upper_date"])
].copy()

control_prior_summary = (
    control_off_prior.groupby(["tip_id", "pcs_off_id", "StartDate"], as_index=False)
    .agg(
        prior_offense_before_start=("offense_date", lambda x: 1 if x.count() > 0 else 0)
    )
)

control_post_summary = (
    control_off_post.groupby(["tip_id", "pcs_off_id", "StartDate"], as_index=False)
    .agg(
        offense_outcome=("offense_date", lambda x: 1 if x.count() > 0 else 0)
    )
)

control_off_outcomes = control_rows.merge(
    control_prior_summary,
    on=["tip_id", "pcs_off_id", "StartDate"],
    how="left"
).merge(
    control_post_summary,
    on=["tip_id", "pcs_off_id", "StartDate"],
    how="left"
)

control_off_outcomes["prior_offense_before_start"] = (
    control_off_outcomes["prior_offense_before_start"].fillna(0)
)

control_off_outcomes["offense_outcome"] = (
    control_off_outcomes["offense_outcome"].fillna(0)
)

control_off_outcomes["recidivism_outcome"] = np.where(
    (control_off_outcomes["prior_offense_before_start"] > 0) &
    (control_off_outcomes["offense_outcome"] > 0),
    1,
    0
)

control_off_outcomes.head()

,tip_id,pcs_off_id,StartDate,prior_offense_before_start,offense_outcome,recidivism_outcome
0,4,1598117.0,2013-08-12,0.0,0.0,0
1,7,1248203.0,2013-07-24,1.0,0.0,0
2,8,1018143.0,2013-08-19,1.0,0.0,0
3,10,1021892.0,2010-07-09,1.0,0.0,0
4,12,1365128.0,2013-02-27,1.0,0.0,0


In [158]:
# --------------------------------------------------
# Block 7: Merge control outcomes back
# --------------------------------------------------

control_outcomes = control_rows.merge(
    control_earn_outcomes,
    on=["tip_id", "pcs_off_id", "StartDate"],
    how="left"
).merge(
    control_off_outcomes[[
        "tip_id", "pcs_off_id", "StartDate", "offense_outcome", "recidivism_outcome"
    ]],
    on=["tip_id", "pcs_off_id", "StartDate"],
    how="left"
)

for col in ["employment_outcome", "earnings_outcome", "offense_outcome", "recidivism_outcome"]:
    if col in control_outcomes.columns:
        control_outcomes[col] = control_outcomes[col].fillna(0)

control_outcomes.head()

,tip_id,pcs_off_id,StartDate,employment_outcome,earnings_outcome,offense_outcome,recidivism_outcome
0,4,1598117.0,2013-08-12,0.0,0.0,0.0,0
1,7,1248203.0,2013-07-24,0.0,0.0,0.0,0
2,8,1018143.0,2013-08-19,0.0,0.0,0.0,0
3,10,1021892.0,2010-07-09,0.0,0.0,0.0,0
4,12,1365128.0,2013-02-27,0.0,0.0,0.0,0


In [159]:
# --------------------------------------------------
# Block 8: Merge all outcomes into matched sample
# --------------------------------------------------

matched_outcome_df = matched_outcome_df.merge(
    participant_outcomes,
    on=["tip_id", "StartDate"],
    how="left",
    suffixes=("", "_participant")
)

matched_outcome_df = matched_outcome_df.merge(
    control_outcomes,
    on=["tip_id", "pcs_off_id", "StartDate"],
    how="left",
    suffixes=("", "_control")
)

# unify participant/control rows into common outcome columns
matched_outcome_df["employment_outcome_final"] = np.where(
    matched_outcome_df["treatment"] == 1,
    matched_outcome_df["employment_outcome"],
    matched_outcome_df["employment_outcome_control"]
)

matched_outcome_df["earnings_outcome_final"] = np.where(
    matched_outcome_df["treatment"] == 1,
    matched_outcome_df["earnings_outcome"],
    matched_outcome_df["earnings_outcome_control"]
)

matched_outcome_df["offense_outcome_final"] = np.where(
    matched_outcome_df["treatment"] == 1,
    matched_outcome_df["offense_outcome"],
    matched_outcome_df["offense_outcome_control"]
)

matched_outcome_df["recidivism_outcome_final"] = np.where(
    matched_outcome_df["treatment"] == 1,
    matched_outcome_df["recidivism_outcome"],
    matched_outcome_df["recidivism_outcome_control"]
)

# replace main columns with unified versions
matched_outcome_df["employment_outcome"] = matched_outcome_df["employment_outcome_final"]
matched_outcome_df["earnings_outcome"] = matched_outcome_df["earnings_outcome_final"]
matched_outcome_df["offense_outcome"] = matched_outcome_df["offense_outcome_final"]
matched_outcome_df["recidivism_outcome"] = matched_outcome_df["recidivism_outcome_final"]

# clean helper columns
drop_cols = [
    "employment_outcome_final", "earnings_outcome_final",
    "offense_outcome_final", "recidivism_outcome_final",
    "employment_outcome_control", "earnings_outcome_control",
    "offense_outcome_control", "recidivism_outcome_control"
]

matched_outcome_df = matched_outcome_df.drop(columns=[c for c in drop_cols if c in matched_outcome_df.columns], errors="ignore")

print("matched_outcome_df shape:", matched_outcome_df.shape)
matched_outcome_df.head()

matched_outcome_df shape: (958, 21)


,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,year_of_birth,...,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip,psm_distance,employment_outcome,earnings_outcome,offense_outcome,recidivism_outcome
0,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,1990.0,...,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
2,8,NaN,participant,1,2013-08-19,2013,0.0,Male,Black,1989.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,10,NaN,participant,1,2010-07-09,2010,0.0,Male,Black,1989.0,...,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,12,NaN,participant,1,2013-02-27,2013,0.0,Male,Black,1987.0,...,3.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [160]:
# --------------------------------------------------
# Block 9: Quick checks
# --------------------------------------------------

print("\nOutcome means by treatment:")
print(
    matched_outcome_df.groupby("treatment")[
        ["employment_outcome", "earnings_outcome", "offense_outcome", "recidivism_outcome"]
    ].mean()
)

print("\nOutcome counts by treatment:")
for col in ["employment_outcome", "offense_outcome", "recidivism_outcome"]:
    print(f"\n{col}")
    print(pd.crosstab(matched_outcome_df["treatment"], matched_outcome_df[col]))


Outcome means by treatment:
           employment_outcome  earnings_outcome  offense_outcome  \
treatment                                                          
0                    0.392484       7742.822612         0.081420   
1                    0.588727       8770.241215         0.200418   

           recidivism_outcome  
treatment                      
0                    0.077244  
1                    0.198330  

Outcome counts by treatment:

employment_outcome
employment_outcome  0.0  1.0
treatment                   
0                   291  188
1                   197  282

offense_outcome
offense_outcome  0.0  1.0
treatment                
0                440   39
1                383   96

recidivism_outcome
recidivism_outcome  0.0  1.0
treatment                   
0                   442   37
1                   384   95


In [161]:
# --------------------------------------------------
# Block 10: Categorical variable distributions by treatment
# --------------------------------------------------

categorical_vars = [
    "unit_type",
    "gender",
    "race",
    "cohort_2022",
    "start_year"
]

for var in categorical_vars:
    print(f"\n===== {var} distribution by treatment =====")
    
    count_table = pd.crosstab(
        matched_outcome_df[var],
        matched_outcome_df["treatment"],
        margins=True,
        dropna=False
    )
    
    # rename columns if treatment values are 0/1
    count_table.columns = [
        "Non-participant" if c == 0 else "Participant" if c == 1 else str(c)
        for c in count_table.columns
    ]
    
    print("\nCounts:")
    print(count_table)

    pct_table = pd.crosstab(
        matched_outcome_df[var],
        matched_outcome_df["treatment"],
        normalize="columns",
        dropna=False
    ) * 100
    
    pct_table.columns = [
        "Non-participant_pct" if c == 0 else "Participant_pct" if c == 1 else str(c)
        for c in pct_table.columns
    ]
    
    print("\nColumn percentages (%):")
    print(pct_table.round(2))


===== unit_type distribution by treatment =====

Counts:
             Non-participant  Participant  All
unit_type                                     
control                  479            0  479
participant                0          479  479
All                      479          479  958

Column percentages (%):
             Non-participant_pct  Participant_pct
unit_type                                        
control                    100.0              0.0
participant                  0.0            100.0

===== gender distribution by treatment =====

Counts:
        Non-participant  Participant  All
gender                                   
Female               24           24   48
Male                455          455  910
All                 479          479  958

Column percentages (%):
        Non-participant_pct  Participant_pct
gender                                      
Female                 5.01             5.01
Male                  94.99            94.99

===== race 

# Main Regression Specification

After constructing the 1:10 matched participant–non-participant sample and defining post-start outcomes, we estimate the main regression model to test whether outcomes improved after 2022.

### Main Specification

For each outcome, we estimate:

\[
Y_i = \alpha + \beta_1 \text{Treatment}_i + \beta_2 \text{Post2022}_i + \beta_3 (\text{Treatment}_i \times \text{Post2022}_i) + \varepsilon_i
\]

where:

- **Treatment** = 1 for TIP participants, 0 for matched non-participants
- **Post2022** = 1 if `StartDate` is on or after January 1, 2022, and 0 otherwise
- **Treatment × Post2022** is the key interaction term

### Interpretation

In this specification:

- **β₁** captures the participant–control difference before 2022
- **β₂** captures the post-2022 shift for the matched non-participant group
- **β₃** captures whether the participant–control difference changes after 2022

The coefficient **β₃** is the main quantity of interest. A positive β₃ indicates improvement in outcomes for participants relative to matched non-participants after 2022, while a negative β₃ indicates relative worsening.

For earnings, a positive interaction term implies improved participant earnings relative to controls after 2022.  
For offense and recidivism outcomes, a negative interaction term implies improvement, because lower values correspond to fewer adverse events.

### Outcomes

We estimate the model separately for:

- **employment_outcome**
- **earnings_outcome**
- **offense_outcome**
- **recidivism_outcome**

### Inference

Because each participant may be matched to multiple controls, standard errors are clustered at the **tip_id** level.

In [162]:
# --------------------------------------------------
# Block 1: Prepare regression dataset
# --------------------------------------------------
import statsmodels.formula.api as smf

reg_df = matched_outcome_df.copy()

# clean core variables
reg_df["StartDate"] = pd.to_datetime(reg_df["StartDate"], errors="coerce")
reg_df["treatment"] = pd.to_numeric(reg_df["treatment"], errors="coerce")
reg_df["cohort_2022"] = pd.to_numeric(reg_df["cohort_2022"], errors="coerce")

outcome_vars = [
    "employment_outcome",
    "earnings_outcome",
    "offense_outcome",
    "recidivism_outcome"
]

for col in outcome_vars:
    reg_df[col] = pd.to_numeric(reg_df[col], errors="coerce")

# keep rows with required fields
reg_df = reg_df.dropna(subset=["tip_id", "treatment", "cohort_2022"])

print("reg_df shape:", reg_df.shape)
print(reg_df[["tip_id", "treatment", "cohort_2022"]].head())

reg_df shape: (958, 21)
   tip_id  treatment  cohort_2022
0       4          1          0.0
1       7          1          0.0
2       8          1          0.0
3      10          1          0.0
4      12          1          0.0


In [163]:
# --------------------------------------------------
# Block 2: Outcome means by treatment and cohort
# --------------------------------------------------

for outcome in outcome_vars:
    print(f"\n===== {outcome} =====")
    print(
        reg_df.groupby(["cohort_2022", "treatment"])[outcome]
        .agg(["count", "mean", "std"])
    )


===== employment_outcome =====
                       count      mean       std
cohort_2022 treatment                           
0.0         0            387  0.387597  0.487832
            1            387  0.560724  0.496941
1.0         0             92  0.413043  0.495079
            1             92  0.706522  0.457851

===== earnings_outcome =====
                       count          mean           std
cohort_2022 treatment                                   
0.0         0            387   7503.225794  16327.215634
            1            387   8200.027623  15066.822543
1.0         0             92   8750.691833  16994.440531
            1             92  11168.857084  16739.157734

===== offense_outcome =====
                       count      mean       std
cohort_2022 treatment                           
0.0         0            387  0.100775  0.301421
            1            387  0.211886  0.409174
1.0         0             92  0.000000  0.000000
            1             92

In [164]:
# --------------------------------------------------
# Block 3: Main regressions with treatment * cohort_2022
# Clustered by tip_id
# --------------------------------------------------

model_emp = smf.ols(
    "employment_outcome ~ treatment * cohort_2022",
    data=reg_df
).fit(cov_type="cluster", cov_kwds={"groups": reg_df["tip_id"]})

print("\n===== Employment Outcome =====")
print(model_emp.summary())

model_earn = smf.ols(
    "earnings_outcome ~ job_right_before_tip + earnings_2yr_preTIP + treatment * cohort_2022",
    data=reg_df
).fit(cov_type="cluster", cov_kwds={"groups": reg_df["tip_id"]})

print("\n===== Earnings Outcome =====")
print(model_earn.summary())

model_off = smf.ols(
    "offense_outcome ~ num_sentences_before_start + num_offenses_before_start + treatment * cohort_2022",
    data=reg_df
).fit(cov_type="cluster", cov_kwds={"groups": reg_df["tip_id"]})

print("\n===== Offense Outcome =====")
print(model_off.summary())

model_recid = smf.ols(
    "recidivism_outcome ~ num_sentences_before_start + num_offenses_before_start + treatment * cohort_2022",
    data=reg_df
).fit(cov_type="cluster", cov_kwds={"groups": reg_df["tip_id"]})

print("\n===== Recidivism Outcome =====")
print(model_recid.summary())


===== Employment Outcome =====
                            OLS Regression Results                            
Dep. Variable:     employment_outcome   R-squared:                       0.045
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     21.81
Date:                Fri, 17 Apr 2026   Prob (F-statistic):           2.97e-13
Time:                        16:00:13   Log-Likelihood:                -672.92
No. Observations:                 958   AIC:                             1354.
Df Residuals:                     954   BIC:                             1373.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------

# Constructing a Relative-Time Panel Dataset

To support panel regression, we reshape the matched participant–non-participant sample into a relative-time panel.

### Panel Unit

Each row in the matched sample is converted into a panel unit:

- for TIP participants: `P_{tip_id}`
- for matched non-participants: `C_{tip_id}_{pcs_off_id}`

This structure is necessary because the same non-participant may be matched to different participants and therefore inherit different anchor dates.

### Time Index

The panel is indexed by **relative year**:

- `year_rel = -2`
- `year_rel = -1`
- `year_rel = 0`
- `year_rel = 1`

where `year_rel = 0` is the first year after `StartDate`, and negative values represent pre-start periods.

### Outcomes in the Panel

For each panel unit and relative year, we construct:

- **earnings_year**: total adjusted earnings in that relative-year window
- **employed_year**: indicator for any positive earnings in that relative-year window
- **offense_count_year**: number of offense records in that relative-year window
- **has_offense_year**: indicator for any offense record in that relative-year window
- **recidivism_year**: indicator for any post-start offense in that relative-year window among units with pre-start offense history

### Use in Panel Regression

The resulting dataset can be used for:

- two-way fixed effects panel regression
- event-study style regressions
- pre/post difference-in-differences specifications

Because controls inherit matched participants’ anchor dates, the panel is aligned on the same relative-time scale for both groups.

In [165]:
# --------------------------------------------------
# Block 1: Build panel base from matched sample
# --------------------------------------------------

panel_base = matched_outcome_df.copy()

panel_base["StartDate"] = pd.to_datetime(panel_base["StartDate"], errors="coerce")
panel_base["treatment"] = pd.to_numeric(panel_base["treatment"], errors="coerce")
panel_base["cohort_2022"] = pd.to_numeric(panel_base["cohort_2022"], errors="coerce")

# Create unique panel unit id
panel_base["unit_id"] = np.where(
    panel_base["treatment"] == 1,
    "P_" + panel_base["tip_id"].astype(str),
    "C_" + panel_base["tip_id"].astype(str) + "_" + panel_base["pcs_off_id"].astype(str)
)

# Keep one row per panel unit
panel_units = panel_base[[
    "unit_id", "tip_id", "pcs_off_id", "unit_type", "treatment",
    "StartDate", "start_year", "cohort_2022", "gender", "race",
    "year_of_birth", "age_of_first_offense",
    "num_sentences_before_start", "any_high_ogs_before_start",
    "num_offenses_before_start", "earnings_2yr_preTIP", "job_right_before_tip",
    "psm_distance"
]].drop_duplicates().copy()

print("panel_units shape:", panel_units.shape)
panel_units.head()

panel_units shape: (958, 18)


,unit_id,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,year_of_birth,age_of_first_offense,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip,psm_distance
0,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0
1,P_7,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,1990.0,19.0,1.0,1.0,1.0,0.0,0.0,0.0
2,P_8,8,NaN,participant,1,2013-08-19,2013,0.0,Male,Black,1989.0,20.0,1.0,0.0,2.0,0.0,0.0,0.0
3,P_10,10,NaN,participant,1,2010-07-09,2010,0.0,Male,Black,1989.0,18.0,2.0,1.0,1.0,0.0,0.0,0.0
4,P_12,12,NaN,participant,1,2013-02-27,2013,0.0,Male,Black,1987.0,23.0,3.0,1.0,1.0,0.0,0.0,0.0


In [166]:
# --------------------------------------------------
# Block 2: Create panel skeleton
# --------------------------------------------------

year_rel_values = [-2, -1, 0, 1]

panel_skeleton = panel_units.assign(key=1).merge(
    pd.DataFrame({"year_rel": year_rel_values, "key": 1}),
    on="key",
    how="outer"
).drop(columns="key")

panel_skeleton["post"] = (panel_skeleton["year_rel"] >= 0).astype(int)

print("panel_skeleton shape:", panel_skeleton.shape)
panel_skeleton.head()

panel_skeleton shape: (3832, 20)


,unit_id,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,year_of_birth,age_of_first_offense,num_sentences_before_start,any_high_ogs_before_start,num_offenses_before_start,earnings_2yr_preTIP,job_right_before_tip,psm_distance,year_rel,post
0,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-2,0
1,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,-1,0
2,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,1982.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,1,1
4,P_7,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,1990.0,19.0,1.0,1.0,1.0,0.0,0.0,0.0,-2,0


In [167]:
# --------------------------------------------------
# Block 3: Participant earnings panel
# --------------------------------------------------

participant_units = panel_units[panel_units["treatment"] == 1][["unit_id", "tip_id", "StartDate"]].copy()

participant_earn_long = participant_units.merge(
    tip_earnings[["tip_id", "year_quarter_dt", "adjusted_earnings"]],
    on="tip_id",
    how="left"
)

participant_earn_long["year_rel"] = (
    (participant_earn_long["year_quarter_dt"] - participant_earn_long["StartDate"]).dt.days / 365.25
).apply(lambda x: np.floor(x) if pd.notna(x) else np.nan)

participant_earn_long["year_rel"] = pd.to_numeric(participant_earn_long["year_rel"], errors="coerce")

participant_earn_panel = (
    participant_earn_long[
        participant_earn_long["year_rel"].isin(year_rel_values)
    ]
    .groupby(["unit_id", "year_rel"], as_index=False)
    .agg(
        earnings_year=("adjusted_earnings", "sum"),
        employed_year=("adjusted_earnings", lambda x: 1 if (x > 0).any() else 0)
    )
)

participant_earn_panel.head()

,unit_id,year_rel,earnings_year,employed_year
0,P_1007,-2.0,83053.111449,1
1,P_1007,-1.0,66352.716196,1
2,P_1007,0.0,15825.501716,1
3,P_1007,1.0,51624.700308,1
4,P_1008,-1.0,0.000000,0


In [168]:
# --------------------------------------------------
# Block 4: Control earnings panel
# --------------------------------------------------

control_units = panel_units[panel_units["treatment"] == 0][["unit_id", "pcs_off_id", "StartDate"]].copy()

control_earn_long = control_units.merge(
    nonparticipant_earnings_long[["pcs_off_id", "year_quarter_dt", "adjusted_earnings"]],
    on="pcs_off_id",
    how="left"
)

control_earn_long["year_rel"] = (
    (control_earn_long["year_quarter_dt"] - control_earn_long["StartDate"]).dt.days / 365.25
).apply(lambda x: np.floor(x) if pd.notna(x) else np.nan)

control_earn_long["year_rel"] = pd.to_numeric(control_earn_long["year_rel"], errors="coerce")

control_earn_panel = (
    control_earn_long[
        control_earn_long["year_rel"].isin(year_rel_values)
    ]
    .groupby(["unit_id", "year_rel"], as_index=False)
    .agg(
        earnings_year=("adjusted_earnings", "sum"),
        employed_year=("adjusted_earnings", lambda x: 1 if (x > 0).any() else 0)
    )
)

control_earn_panel.head()

,unit_id,year_rel,earnings_year,employed_year
0,C_1007_1068743.0,-2.0,65143.991988,1
1,C_1007_1068743.0,-1.0,29625.834457,1
2,C_1007_1068743.0,0.0,56017.882361,1
3,C_1007_1068743.0,1.0,2096.297481,1
4,C_1008_1136980.0,-2.0,0.000000,0


In [169]:
# --------------------------------------------------
# Block 5: Participant offense panel
# --------------------------------------------------

participant_off_long = participant_units.merge(
    combined_offense_dates[["tip_id", "offense_date"]],
    on="tip_id",
    how="left"
)

participant_off_long["year_rel"] = (
    (participant_off_long["offense_date"] - participant_off_long["StartDate"]).dt.days / 365.25
).apply(lambda x: np.floor(x) if pd.notna(x) else np.nan)

participant_off_long["year_rel"] = pd.to_numeric(participant_off_long["year_rel"], errors="coerce")

# yearly offense outcomes
participant_off_panel = (
    participant_off_long[
        participant_off_long["year_rel"].isin(year_rel_values)
    ]
    .groupby(["unit_id", "year_rel"], as_index=False)
    .agg(
        offense_count_year=("offense_date", "count"),
        has_offense_year=("offense_date", lambda x: 1 if x.count() > 0 else 0)
    )
)

# prior / cumulative offense indicators by year
participant_history_rows = []

for unit_id in participant_units["unit_id"].unique():
    unit_off = participant_off_long.loc[
        participant_off_long["unit_id"] == unit_id,
        "year_rel"
    ].dropna()

    for yr in year_rel_values:
        has_prior = int((unit_off < yr).any())   # before this year only
        has_by = int((unit_off <= yr).any())     # this year and before
        participant_history_rows.append({
            "unit_id": unit_id,
            "year_rel": yr,
            "has_prior_offense_by_year": has_prior,
            "has_offense_by_year": has_by
        })

participant_history_panel = pd.DataFrame(participant_history_rows)

participant_off_panel = participant_history_panel.merge(
    participant_off_panel,
    on=["unit_id", "year_rel"],
    how="left"
)

participant_off_panel["offense_count_year"] = participant_off_panel["offense_count_year"].fillna(0)
participant_off_panel["has_offense_year"] = participant_off_panel["has_offense_year"].fillna(0)
participant_off_panel["has_prior_offense_by_year"] = participant_off_panel["has_prior_offense_by_year"].fillna(0)
participant_off_panel["has_offense_by_year"] = participant_off_panel["has_offense_by_year"].fillna(0)

participant_off_panel.head()

,unit_id,year_rel,has_prior_offense_by_year,has_offense_by_year,offense_count_year,has_offense_year
0,P_4,-2,0,0,0.0,0.0
1,P_4,-1,0,0,0.0,0.0
2,P_4,0,0,0,0.0,0.0
3,P_4,1,0,0,0.0,0.0
4,P_7,-2,1,1,0.0,0.0


In [170]:
# --------------------------------------------------
# Block 6: Control offense panel
# --------------------------------------------------

control_off_long = control_units.merge(
    nonparticipant_pcs_long[["pcs_off_id", "DOF"]],
    on="pcs_off_id",
    how="left"
).rename(columns={"DOF": "offense_date"})

control_off_long["year_rel"] = (
    (control_off_long["offense_date"] - control_off_long["StartDate"]).dt.days / 365.25
).apply(lambda x: np.floor(x) if pd.notna(x) else np.nan)

control_off_long["year_rel"] = pd.to_numeric(control_off_long["year_rel"], errors="coerce")

# yearly offense outcomes
control_off_panel = (
    control_off_long[
        control_off_long["year_rel"].isin(year_rel_values)
    ]
    .groupby(["unit_id", "year_rel"], as_index=False)
    .agg(
        offense_count_year=("offense_date", "count"),
        has_offense_year=("offense_date", lambda x: 1 if x.count() > 0 else 0)
    )
)

# prior / cumulative offense indicators by year
control_history_rows = []

for unit_id in control_units["unit_id"].unique():
    unit_off = control_off_long.loc[
        control_off_long["unit_id"] == unit_id,
        "year_rel"
    ].dropna()

    for yr in year_rel_values:
        has_prior = int((unit_off < yr).any())   # before this year only
        has_by = int((unit_off <= yr).any())     # this year and before
        control_history_rows.append({
            "unit_id": unit_id,
            "year_rel": yr,
            "has_prior_offense_by_year": has_prior,
            "has_offense_by_year": has_by
        })

control_history_panel = pd.DataFrame(control_history_rows)

control_off_panel = control_history_panel.merge(
    control_off_panel,
    on=["unit_id", "year_rel"],
    how="left"
)

control_off_panel["offense_count_year"] = control_off_panel["offense_count_year"].fillna(0)
control_off_panel["has_offense_year"] = control_off_panel["has_offense_year"].fillna(0)
control_off_panel["has_prior_offense_by_year"] = control_off_panel["has_prior_offense_by_year"].fillna(0)
control_off_panel["has_offense_by_year"] = control_off_panel["has_offense_by_year"].fillna(0)

control_off_panel.head()

,unit_id,year_rel,has_prior_offense_by_year,has_offense_by_year,offense_count_year,has_offense_year
0,C_4_1598117.0,-2,0,0,0.0,0.0
1,C_4_1598117.0,-1,0,0,0.0,0.0
2,C_4_1598117.0,0,0,0,0.0,0.0
3,C_4_1598117.0,1,0,0,0.0,0.0
4,C_7_1248203.0,-2,1,1,0.0,0.0


In [171]:
# --------------------------------------------------
# Block 7: Combine event panels
# --------------------------------------------------

earn_panel = pd.concat(
    [participant_earn_panel, control_earn_panel],
    ignore_index=True
)

off_panel = pd.concat(
    [participant_off_panel, control_off_panel],
    ignore_index=True
)

panel_df = panel_skeleton.merge(
    earn_panel,
    on=["unit_id", "year_rel"],
    how="left"
).merge(
    off_panel,
    on=["unit_id", "year_rel"],
    how="left"
)

# Fill missing outcomes with zero
for col in ["earnings_year", "employed_year", "offense_count_year", "has_offense_year", "has_prior_offense_by_year", "has_offense_by_year"]:
    if col in panel_df.columns:
        panel_df[col] = panel_df[col].fillna(0)

panel_df.head()

,unit_id,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,...,job_right_before_tip,psm_distance,year_rel,post,earnings_year,employed_year,has_prior_offense_by_year,has_offense_by_year,offense_count_year,has_offense_year
0,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,0.0,0.0,-2,0,0.0,0.0,0,0,0.0,0.0
1,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,0.0,0.0,-1,0,0.0,0.0,0,0,0.0,0.0
2,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,0.0,0.0,0,1,0.0,0.0,0,0,0.0,0.0
3,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,0.0,0.0,1,1,0.0,0.0,0,0,0.0,0.0
4,P_7,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,...,0.0,0.0,-2,0,0.0,0.0,1,1,0.0,0.0


In [172]:
# --------------------------------------------------
# Block 8: Construct recidivism_year
# --------------------------------------------------

panel_df["has_prior_offense_before_start"] = np.where(
    panel_df["num_offenses_before_start"] > 0,
    1,
    0
)

panel_df["recidivism_year"] = np.where(
    (panel_df["has_prior_offense_before_start"] == 1) &
    (panel_df["year_rel"] >= 0) &
    (panel_df["has_offense_by_year"] > 0),
    1,
    0
)

panel_df.head()

,unit_id,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,...,year_rel,post,earnings_year,employed_year,has_prior_offense_by_year,has_offense_by_year,offense_count_year,has_offense_year,has_prior_offense_before_start,recidivism_year
0,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,-2,0,0.0,0.0,0,0,0.0,0.0,0,0
1,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,-1,0,0.0,0.0,0,0,0.0,0.0,0,0
2,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,0,1,0.0,0.0,0,0,0.0,0.0,0,0
3,P_4,4,NaN,participant,1,2013-08-12,2013,0.0,Male,Black,...,1,1,0.0,0.0,0,0,0.0,0.0,0,0
4,P_7,7,NaN,participant,1,2013-07-24,2013,0.0,Male,Black,...,-2,0,0.0,0.0,1,1,0.0,0.0,1,0


In [173]:
# --------------------------------------------------
# Block 9: Final panel formatting
# --------------------------------------------------

panel_df = panel_df.sort_values(["unit_id", "year_rel"]).reset_index(drop=True)

# Optional calendar year corresponding to each relative year
panel_df["calendar_year"] = panel_df["start_year"] + panel_df["year_rel"]

print("panel_df shape:", panel_df.shape)
print("Unique panel units:", panel_df["unit_id"].nunique())
print("Rows per unit (should be 4 if balanced):")
print(panel_df.groupby("unit_id").size().value_counts().sort_index())

panel_df.head(20)

panel_df shape: (3832, 29)
Unique panel units: 958
Rows per unit (should be 4 if balanced):
4    958
Name: count, dtype: int64


,unit_id,tip_id,pcs_off_id,unit_type,treatment,StartDate,start_year,cohort_2022,gender,race,...,post,earnings_year,employed_year,has_prior_offense_by_year,has_offense_by_year,offense_count_year,has_offense_year,has_prior_offense_before_start,recidivism_year,calendar_year
0,C_1007_1068743.0,1007,1068743.0,control,0,2019-06-17,2019,0.0,Male,White,...,0,65143.991988,1.0,1,1,0.0,0.0,1,0,2017
1,C_1007_1068743.0,1007,1068743.0,control,0,2019-06-17,2019,0.0,Male,White,...,0,29625.834457,1.0,1,1,0.0,0.0,1,0,2018
2,C_1007_1068743.0,1007,1068743.0,control,0,2019-06-17,2019,0.0,Male,White,...,1,56017.882361,1.0,1,1,0.0,0.0,1,1,2019
3,C_1007_1068743.0,1007,1068743.0,control,0,2019-06-17,2019,0.0,Male,White,...,1,2096.297481,1.0,1,1,0.0,0.0,1,1,2020
4,C_1008_1136980.0,1008,1136980.0,control,0,2019-06-17,2019,0.0,Female,White,...,0,0.000000,0.0,1,1,0.0,0.0,1,0,2017
5,C_1008_1136980.0,1008,1136980.0,control,0,2019-06-17,2019,0.0,Female,White,...,0,0.000000,0.0,1,1,0.0,0.0,1,0,2018
6,C_1008_1136980.0,1008,1136980.0,control,0,2019-06-17,2019,0.0,Female,White,...,1,0.000000,0.0,1,1,0.0,0.0,1,1,2019
7,C_1008_1136980.0,1008,1136980.0,control,0,2019-06-17,2019,0.0,Female,White,...,1,0.000000,0.0,1,1,0.0,0.0,1,1,2020
8,C_1012_1031497.0,1012,1031497.0,control,0,2019-06-17,2019,0.0,Male,White,...,0,0.000000,0.0,1,1,0.0,0.0,1,0,2017
9,C_1012_1031497.0,1012,1031497.0,control,0,2019-06-17,2019,0.0,Male,White,...,0,29849.192414,1.0,1,1,0.0,0.0,1,0,2018


In [174]:
# --------------------------------------------------
# Block 10: Quick checks
# --------------------------------------------------

print("\nOutcome means by treatment and relative year:")
print(
    panel_df.groupby(["treatment", "year_rel"])[
        ["earnings_year", "employed_year", "offense_count_year", "has_offense_year", "has_offense_by_year", "has_prior_offense_by_year", "recidivism_year"]
    ].mean()
)

print("\nCounts by treatment and relative year:")
print(
    panel_df.groupby(["treatment", "year_rel"])
    .size()
    .unstack(fill_value=0)
)


Outcome means by treatment and relative year:
                    earnings_year  employed_year  offense_count_year  \
treatment year_rel                                                     
0         -2          3039.026512       0.233820            0.296451   
          -1          4676.084241       0.340292            0.179541   
           0          5257.892797       0.348643            0.077244   
           1          5150.623788       0.334029            0.112735   
1         -2          3757.859328       0.294363            0.336117   
          -1          4573.681497       0.390397            0.275574   
           0          5129.715733       0.517745            0.210856   
           1          7397.314193       0.469729            0.123173   

                    has_offense_year  has_offense_by_year  \
treatment year_rel                                          
0         -2                0.131524             0.864301   
          -1                0.106472             

In [175]:
# --------------------------------------------------
# Block 11: create panel dataset for recidivism which only includes people with offense before start
# --------------------------------------------------
recid_panel_df = panel_df[panel_df["has_prior_offense_before_start"] == 1].copy()
print("recid_panel_df shape:", recid_panel_df.shape)

recid_panel_df shape: (3568, 29)


# Panel Regression Framework for the Matched Relative-Time Sample

We estimate panel models using the matched relative-time dataset. Each matched unit is observed over relative years around the participant’s TIP start date:

- `year_rel = -2`
- `year_rel = -1`
- `year_rel = 0`
- `year_rel = 1`

For TIP participants, `StartDate` is the observed TIP start date.  
For matched non-participants, `StartDate` is the inherited anchor date from the matched participant.

The panel includes both employment and criminal-justice outcomes measured at the relative-year level.

### Core Variables

- **$Treatment_i$**: indicator for whether unit $i$ is a TIP participant
- **$Post_t$**: indicator for whether relative year $t$ is on or after treatment start
- **$Cohort\_TIP_i$**: indicator for whether the matched participant belongs to the post-2022 cohort
- **$Treatment_i \times Post_t$**: participant–control post-start effect for the pre-2022 cohort
- **$Treatment_i \times Post_t \times Cohort\_TIP_i$**: additional post-start effect for the post-2022 cohort

### Model Specification

The estimation equation is:

$$Y_{it} = \alpha_i + \gamma_t + \beta_1 (Treatment_i \times Post_t) + \beta_2 (Treatment_i \times Post_t \times Cohort\_TIP_i) + \epsilon_{it}$$

All fixed effects models include:

- **$\alpha_i$**: individual fixed effects
- **$\gamma_t$**: relative-time fixed effects

This specification allows us to ask whether TIP participants change relative to matched non-participants after treatment start, and whether that relative change differs for the post-2022 cohort.

## Two-Way Fixed Effects Model for Employment and Earnings

We estimate two-way fixed effects models for annual employment and annual earnings outcomes.

#### Employment

$$
\text{Employed}_{it} = \alpha_i + \gamma_t + \beta (\text{Treatment}_i \times \text{Post}_t) + \delta (\text{Treatment}_i \times \text{Post}_t \times \text{Cohort\_TIP}_i) + \zeta \, \text{HasOffenseByYear}_{it} + \epsilon_{it}
$$

**Where:**

- **$\text{Employed}_{it}$**: a binary indicator for whether individual $i$ has any positive earnings in relative year $t$
- **$\alpha_i$**: the individual fixed effect
- **$\gamma_t$**: the relative-time fixed effect
- **$\text{Treatment}_i \times \text{Post}_t$**: captures the post-start participant–control difference for the pre-2022 cohort
- **$\text{Treatment}_i \times \text{Post}_t \times \text{Cohort\_TIP}_i$**: captures whether that post-start participant–control difference changes for the post-2022 cohort
- **$\beta$**: the coefficient on the participant post-start effect
- **$\delta$**: the coefficient on the additional post-2022 participant post-start effect
- **$\epsilon_{it}$**: the error term

#### Earnings

$$
\text{Earnings}_{it} = \alpha_i + \gamma_t + \beta (\text{Treatment}_i \times \text{Post}_t) + \delta (\text{Treatment}_i \times \text{Post}_t \times \text{Cohort\_TIP}_i) + \zeta \, \text{HasOffenseByYear}_{it} + \epsilon_{it}
$$

**Where:**

- **$\text{Earnings}_{it}$**: the annual earnings outcome for individual $i$ in relative year $t$
- In the main specification, this is measured as annual adjusted earnings
- In a robustness check, we also estimate the same model using **log annual earnings**, defined as $\log(1 + \text{Earnings}_{it})$

---

## Two-Way Fixed Effects Model for Criminal Justice Outcomes

We estimate two-way fixed effects models for yearly criminal-justice interactions.

### Any Offense in Year

$$
\text{HasOffenseYear}_{it} = \alpha_i + \gamma_t + \beta (\text{Treatment}_i \times \text{Post}_t) + \delta (\text{Treatment}_i \times \text{Post}_t \times \text{Cohort\_TIP}_i) + \zeta \, \text{HasPriorOffenseByYear}_{it} + \epsilon_{it}
$$

### Offense Count in Year

$$
\text{OffenseCountYear}_{it} = \alpha_i + \gamma_t + \beta (\text{Treatment}_i \times \text{Post}_t) + \delta (\text{Treatment}_i \times \text{Post}_t \times \text{Cohort\_TIP}_i) + \zeta \, \text{HasPriorOffenseByYear}_{it} + \epsilon_{it}
$$

**Where:**

- **$\text{HasOffenseYear}_{it}$**: a binary indicator for whether individual $i$ has any offense record in relative year $t$
- **$\text{OffenseCountYear}_{it}$**: the number of offense records in relative year $t$
- **$\text{HasPriorOffenseByYear}_{it}$**: a binary indicator for whether individual $i$ has any offense history before relative year $t$, excluding offenses occurring in year $t$
- **$\alpha_i$**: the individual fixed effect
- **$\gamma_t$**: the relative-time fixed effect
- **$\beta$**, **$\delta$**: coefficients representing cohort-specific treatment effects
- **$\zeta$**: captures the association between prior offense history and current-year offending
- **$\epsilon_{it}$**: the error term

---

## Conditional Logit Model for Recidivism

Recidivism is modeled separately using a conditional logit specification.

This is because:
1. The outcome is binary
2. Recidivism is only meaningfully defined for unit-years with prior offense history
3. The model conditions on within-unit variation over time

### Outcome Definition

- **$\text{RecidivismYear}_{it}$**: equals 1 if individual $i$ has an offense in relative year $t$ **and** has prior offense history before year $t$
- **$\text{HasPriorOffenseByYear}_{it}$**: indicates whether individual $i$ has any offense history before relative year $t$, excluding offenses in the current year

### Specification

$$
Pr(\text{RecidivismYear}_{it} = 1 \mid \alpha_i) = \text{logit}^{-1} \left[ \alpha_i + \gamma_t + \beta (\text{Treatment}_i \times \text{Post}_t) + \delta (\text{Treatment}_i \times \text{Post}_t \times \text{Cohort\_TIP}_i) + \zeta \, \text{HasPriorOffenseByYear}_{it} \right]
$$

**Where:**

- **$\alpha_i$**: the individual-specific effect conditioned out in the conditional logit model
- **$\gamma_t$**: the relative-time effect
- **$\beta$**, **$\delta$**, **$\zeta$**: model coefficients as defined in the linear specifications

In [176]:
recid_panel_df.columns

Index(['unit_id', 'tip_id', 'pcs_off_id', 'unit_type', 'treatment',
       'StartDate', 'start_year', 'cohort_2022', 'gender', 'race',
       'year_of_birth', 'age_of_first_offense', 'num_sentences_before_start',
       'any_high_ogs_before_start', 'num_offenses_before_start',
       'earnings_2yr_preTIP', 'job_right_before_tip', 'psm_distance',
       'year_rel', 'post', 'earnings_year', 'employed_year',
       'has_prior_offense_by_year', 'has_offense_by_year',
       'offense_count_year', 'has_offense_year',
       'has_prior_offense_before_start', 'recidivism_year', 'calendar_year'],
      dtype='object')

In [177]:
# --------------------------------------------------
# Block 1: Prepare panel data for panel FE models
# --------------------------------------------------

from linearmodels.panel import PanelOLS
from statsmodels.discrete.conditional_models import ConditionalLogit

fe_data = panel_df.copy()
fe_data_recid = recid_panel_df.copy()

# numeric conversion
for col in [
    "tip_id",
    "year_rel",
    "earnings_year",
    "employed_year",
    "has_offense_year",
    "recidivism_year",
    "treatment",
    "post",
    "cohort_2022",
    "has_offense_by_year",
    "has_prior_offense_by_year"
]:
    fe_data[col] = pd.to_numeric(fe_data[col], errors="coerce")
    fe_data_recid[col] = pd.to_numeric(fe_data_recid[col], errors="coerce")

# interaction terms
fe_data["treatment_post"] = fe_data["treatment"] * fe_data["post"]
fe_data_recid["treatment_post"] = fe_data_recid["treatment"] * fe_data_recid["post"]
fe_data["treatment_post_cohort"] = (
    fe_data["treatment"] * fe_data["post"] * fe_data["cohort_2022"]
)
fe_data_recid["treatment_post_cohort"] = (
    fe_data_recid["treatment"] * fe_data_recid["post"] * fe_data_recid["cohort_2022"]
)

# log earnings
fe_data["log_earnings_year"] = np.log1p(fe_data["earnings_year"])
fe_data_recid["log_earnings_year"] = np.log1p(fe_data_recid["earnings_year"])

# set panel index like your example
fe_data = fe_data.set_index(["tip_id", "year_rel"]).sort_index()
fe_data_recid = fe_data_recid.set_index(["tip_id", "year_rel"]).sort_index()

print(
    fe_data[[
        "earnings_year", "log_earnings_year",
        "treatment_post", "treatment_post_cohort",
        "has_offense_by_year", "has_prior_offense_by_year"
    ]].head(10)
)

                 earnings_year  log_earnings_year  treatment_post  \
tip_id year_rel                                                     
4      -2                  0.0                0.0               0   
       -2                  0.0                0.0               0   
       -1                  0.0                0.0               0   
       -1                  0.0                0.0               0   
        0                  0.0                0.0               0   
        0                  0.0                0.0               1   
        1                  0.0                0.0               0   
        1                  0.0                0.0               1   
7      -2                  0.0                0.0               0   
       -2                  0.0                0.0               0   

                 treatment_post_cohort  has_offense_by_year  \
tip_id year_rel                                               
4      -2                          0.0       

In [178]:
# --------------------------------------------------
# Block 2: FE model for earnings_year
# --------------------------------------------------

model_fe_earn = PanelOLS.from_formula(
    formula="""
    earnings_year ~ 1
    + treatment_post
    + treatment_post_cohort
    + has_offense_by_year
    + EntityEffects
    + TimeEffects
    """,
    data=fe_data
)

result_fe_earn = model_fe_earn.fit(
    cov_type="clustered",
    cluster_entity=True,
    cluster_time=True
)

print("\n===== FE: earnings_year =====")
print(result_fe_earn.summary)


===== FE: earnings_year =====
                          PanelOLS Estimation Summary                           
Dep. Variable:          earnings_year   R-squared:                        0.0036
Estimator:                   PanelOLS   R-squared (Between):             -0.0088
No. Observations:                3832   R-squared (Within):               0.0087
Date:                Fri, Apr 17 2026   R-squared (Overall):              0.0020
Time:                        16:00:32   Log-likelihood                -4.016e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      4.0367
Entities:                         479   P-value                           0.0071
Avg Obs:                       8.0000   Distribution:                  F(3,3347)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             0.59

In [179]:
# --------------------------------------------------
# Block 3: FE model for log_earnings_year
# --------------------------------------------------

model_fe_log_earn = PanelOLS.from_formula(
    formula="""
    log_earnings_year ~ 1
    + treatment_post
    + treatment_post_cohort
    + has_offense_by_year
    + EntityEffects
    + TimeEffects
    """,
    data=fe_data
)

result_fe_log_earn = model_fe_log_earn.fit(
    cov_type="clustered",
    cluster_entity=True,
    cluster_time=True
)

print("\n===== FE: log_earnings_year =====")
print(result_fe_log_earn.summary)


===== FE: log_earnings_year =====
                          PanelOLS Estimation Summary                           
Dep. Variable:      log_earnings_year   R-squared:                        0.0258
Estimator:                   PanelOLS   R-squared (Between):             -0.0196
No. Observations:                3832   R-squared (Within):               0.0439
Date:                Fri, Apr 17 2026   R-squared (Overall):              0.0141
Time:                        16:00:34   Log-likelihood                   -9706.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      29.565
Entities:                         479   P-value                           0.0000
Avg Obs:                       8.0000   Distribution:                  F(3,3347)
Min Obs:                       8.0000                                           
Max Obs:                       8.0000   F-statistic (robust):             

In [180]:
# --------------------------------------------------
# Block 4: Helper for conditional logit
# --------------------------------------------------

def run_conditional_logit(df, yvar, control_var):
    temp = df.reset_index().copy()

    keep_cols = [
        "tip_id", "year_rel", yvar,
        "treatment_post", "treatment_post_cohort",
        control_var
    ]
    temp = temp[keep_cols].dropna().copy()

    # only keep groups with within-group variation in outcome
    valid_groups = (
        temp.groupby("tip_id")[yvar]
        .nunique()
        .reset_index(name="n_unique")
    )
    valid_groups = valid_groups.loc[valid_groups["n_unique"] > 1, "tip_id"]

    temp = temp[temp["tip_id"].isin(valid_groups)].copy()

    # time fixed effects via year_rel dummies
    year_dummies = pd.get_dummies(
        temp["year_rel"],
        prefix="year_rel",
        drop_first=True,
        dtype=float
    )

    X = pd.concat(
        [
            temp[["treatment_post", "treatment_post_cohort", control_var]].astype(float),
            year_dummies
        ],
        axis=1
    )

    y = temp[yvar].astype(int)
    groups = temp["tip_id"]

    model = ConditionalLogit(y, X, groups=groups)
    result = model.fit(disp=False)

    return result, temp

In [181]:
# --------------------------------------------------
# Block 5: Conditional logit for employed_year
# include has_offense_by_year as control
# --------------------------------------------------

result_clogit_emp, emp_df_used = run_conditional_logit(
    fe_data,
    yvar="employed_year",
    control_var="has_offense_by_year"
)

print("\n===== Conditional Logit: employed_year =====")
print(result_clogit_emp.summary())
print("\nRows used:", len(emp_df_used), "| tip_id groups used:", emp_df_used["tip_id"].nunique())


===== Conditional Logit: employed_year =====
                  Conditional Logit Model Regression Results                  
Dep. Variable:          employed_year   No. Observations:                 2592
Model:               ConditionalLogit   No. groups:                        324
Log-Likelihood:               -960.84   Min group size:                      8
Method:                          BFGS   Max group size:                      8
Date:                Fri, 17 Apr 2026   Mean group size:                   8.0
Time:                        16:00:41                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
treatment_post            1.3489      0.141      9.546      0.000       1.072       1.626
treatment_post_cohort    -1.0457      0.239     -4.384      0.000      -1.513      -0.578
has_offense_by_year      -0.3099      0.4

In [182]:
# --------------------------------------------------
# Block 6: Conditional logit for has_offense_year
# include has_prior_offense_by_year as control
# --------------------------------------------------

result_clogit_off, off_df_used = run_conditional_logit(
    fe_data,
    yvar="has_offense_year",
    control_var="has_prior_offense_by_year"
)

print("\n===== Conditional Logit: has_offense_year =====")
print(result_clogit_off.summary())
print("\nRows used:", len(off_df_used), "| tip_id groups used:", off_df_used["tip_id"].nunique())


===== Conditional Logit: has_offense_year =====
                  Conditional Logit Model Regression Results                  
Dep. Variable:       has_offense_year   No. Observations:                 2184
Model:               ConditionalLogit   No. groups:                        273
Log-Likelihood:               -696.91   Min group size:                      8
Method:                          BFGS   Max group size:                      8
Date:                Fri, 17 Apr 2026   Mean group size:                   8.0
Time:                        16:00:43                                         
                                coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------
treatment_post                1.0372      0.182      5.684      0.000       0.680       1.395
treatment_post_cohort         0.3137      0.336      0.934      0.350      -0.345       0.972
has_prior_offense_by_y

In [186]:
# --------------------------------------------------
# Block 7: Conditional logit for recidivism_year
# include has_prior_offense_by_year as control
# --------------------------------------------------

result_clogit_recid, recid_df_used = run_conditional_logit(
    fe_data_recid,
    yvar="recidivism_year",
    control_var="has_prior_offense_by_year"
)

print("\n===== Conditional Logit: recidivism_year =====")
print(result_clogit_recid.summary())
print("\nRows used:", len(recid_df_used), "| tip_id groups used:", recid_df_used["tip_id"].nunique())

c:\Program Files\Python312\Lib\site-packages\statsmodels\base\model.py:595: HessianInversionWarning: Inverting hessian failed, no bse or cov_params available
  warnings.warn('Inverting hessian failed, no bse or cov_params '


ValueError: need covariance of parameters for computing (unnormalized) covariances

In [187]:
fe_data.columns

Index(['unit_id', 'pcs_off_id', 'unit_type', 'treatment', 'StartDate',
       'start_year', 'cohort_2022', 'gender', 'race', 'year_of_birth',
       'age_of_first_offense', 'num_sentences_before_start',
       'any_high_ogs_before_start', 'num_offenses_before_start',
       'earnings_2yr_preTIP', 'job_right_before_tip', 'psm_distance', 'post',
       'earnings_year', 'employed_year', 'has_prior_offense_by_year',
       'has_offense_by_year', 'offense_count_year', 'has_offense_year',
       'has_prior_offense_before_start', 'recidivism_year', 'calendar_year',
       'treatment_post', 'treatment_post_cohort', 'log_earnings_year'],
      dtype='object')

In [188]:
fe_data_recid.columns

Index(['unit_id', 'pcs_off_id', 'unit_type', 'treatment', 'StartDate',
       'start_year', 'cohort_2022', 'gender', 'race', 'year_of_birth',
       'age_of_first_offense', 'num_sentences_before_start',
       'any_high_ogs_before_start', 'num_offenses_before_start',
       'earnings_2yr_preTIP', 'job_right_before_tip', 'psm_distance', 'post',
       'earnings_year', 'employed_year', 'has_prior_offense_by_year',
       'has_offense_by_year', 'offense_count_year', 'has_offense_year',
       'has_prior_offense_before_start', 'recidivism_year', 'calendar_year',
       'treatment_post', 'treatment_post_cohort', 'log_earnings_year'],
      dtype='object')